In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [1/8] 環境セットアップ
# MIT License / Copyright (c) 2026 grapodotnet
# ------------------------------------------------------------
# ランタイム起動後に1回だけ実行する。以降のセルは再実行しても
# このセルを流し直す必要はない。
# ============================================================
!apt-get install -y libcairo2-dev > /dev/null
!apt-get install -y fonts-noto-core fonts-noto-extra > /dev/null
!pip install -q requests beautifulsoup4 pandas tabulate lxml google-api-python-client google-auth-httplib2 google-auth-oauthlib japanize-matplotlib cairosvg playwright nest_asyncio pillow
!python -m playwright install --with-deps chromium > /dev/null

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [2/8] 共通基盤（import / ログ / 実行レポート）
# ------------------------------------------------------------
# Ver.9.0.0 の主な変更:
#  1. 単一セル2,114行を8セルに分割。短縮しすぎた識別子を展開し、
#     print をすべて logger に統一した。
#  2. 実行結果サマリ(RunReport)を追加。表構成の想定外や別名未解決を
#     黙って素通ししていた箇所を警告として拾い上げるようにした。
#  3. Drive の一時画像を専用フォルダに入れ、Docs 出力後に公開権限を
#     剥奪する。古い一時画像はゴミ箱へ退避する。
#  4. CBPランキングのカテゴリ(J1/J2/J3)を切り替え可能にした。auto は
#     ランキング表に自チームが載っているカテゴリを実測で判定する。
#  5. FootballLabScraper.file_date が例外時に "unknown_date" のまま
#     残っていた不具合を修正。
# ------------------------------------------------------------
# Ver.8.0 の主な変更（参考）:
#  - CBPランキング7ページを1回だけ取得して今節/次節で共有
#  - 試合HTML/BeautifulSoupをタイムライン・シュートチャートで再利用
#  - Google Docsの末尾indexをローカル管理し documents.get() を削減
#  - requestsに一時エラー向けRetryを追加
#  - PlaywrightはSVG生成要素を直接待つ方式に変更
# ============================================================
from __future__ import annotations

import asyncio
import io
import logging
import re
import sys
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from enum import Enum
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Final, List, Optional, Sequence, Tuple, Union
from urllib.parse import parse_qs, urljoin, urlparse

import cairosvg
import nest_asyncio
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from PIL import Image
from playwright.async_api import Error as PlaywrightError, TimeoutError as PlaywrightTimeoutError, async_playwright
import matplotlib.pyplot as plt
import japanize_matplotlib

try:
    from google.colab import auth
    from google.auth import default
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

VERSION: Final[str] = "9.0.0"

logger = logging.getLogger("konwenga")

_QUIET_LOGGERS: Final[Tuple[str, ...]] = (
    "googleapiclient.discovery_cache",
    "google_auth_httplib2",
    "matplotlib",
    "matplotlib.font_manager",
    "PIL",
    "urllib3",
)


def setup_logging(level: str = "INFO") -> None:
    """Ver.9 では print を廃止して logger に集約したため、出力先と体裁を
    ここで一本化する。Colab の再実行でハンドラが重複しないよう都度張り直す。"""
    root_logger = logging.getLogger()
    for handler in list(root_logger.handlers):
        root_logger.removeHandler(handler)
    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    root_logger.addHandler(stream_handler)
    resolved_level = getattr(logging, str(level).strip().upper(), logging.INFO)
    root_logger.setLevel(resolved_level)
    logger.setLevel(resolved_level)
    for name in _QUIET_LOGGERS:
        logging.getLogger(name).setLevel(logging.WARNING)


setup_logging("INFO")
nest_asyncio.apply()


# ------------------------------------------------------------
# 実行結果サマリ
# ------------------------------------------------------------
class StepStatus(str, Enum):
    OK = "OK"
    NG = "NG"
    SKIP = "SKIP"


@dataclass
class StepResult:
    name: str
    status: StepStatus
    detail: str = ""


class RunReport:
    """成果物ごとの成否と警告を集約する。

    Ver.8 までは途中で何が欠けても最後に「完了」とだけ出ていたため、
    記事を書き始めてから欠落に気づくことがあった。ここに集約して
    実行の最後に必ず一覧を出す。"""

    def __init__(self) -> None:
        self.steps: List[StepResult] = []
        self.warnings: List[str] = []
        self.document_url: str = ""

    def record(self, name: str, status: StepStatus, detail: str = "") -> None:
        self.steps.append(StepResult(name=name, status=status, detail=detail))

    def ok(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.OK, detail)

    def ng(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.NG, detail)

    def skip(self, name: str, detail: str = "") -> None:
        self.record(name, StepStatus.SKIP, detail)

    def warn(self, message: str) -> None:
        logger.warning(message)
        self.warnings.append(message)

    @property
    def failed_steps(self) -> List[StepResult]:
        return [step for step in self.steps if step.status is StepStatus.NG]

    def render(self) -> str:
        name_width = max([display_width(step.name) for step in self.steps] + [16])
        lines = ["", "=== 実行結果サマリ ==="]
        for step in self.steps:
            detail = f"  {step.detail}" if step.detail else ""
            lines.append(f"{pad_display(step.name, name_width)}  {step.status.value:<4}{detail}")
        if self.document_url:
            lines.append(f"{pad_display('Docs URL', name_width)}  {'':<4}  {self.document_url}")
        if self.warnings:
            lines.append("")
            lines.append(f"警告 {len(self.warnings)}件:")
            lines.extend(f"  - {message}" for message in self.warnings)
        else:
            lines.append("")
            lines.append("警告なし")
        failed = self.failed_steps
        lines.append("")
        if failed:
            lines.append(f"失敗 {len(failed)}件: {', '.join(step.name for step in failed)}")
            lines.append("記事に使う前に、上記の成果物が欠けていないか確認してください。")
        elif self.warnings:
            lines.append("失敗はありませんが、警告の内容が記事に影響しないか確認してください。")
        else:
            lines.append("すべての成果物を生成しました。")
        lines.append("=" * 22)
        return "\n".join(lines)

    def emit(self) -> None:
        logger.info(self.render())


def display_width(text: str) -> int:
    """全角を2、半角を1として数える。ログの列を目視で揃えるため。"""
    return sum(2 if unicodedata.east_asian_width(char) in ("W", "F") else 1 for char in str(text))


def pad_display(text: str, width: int, align: str = "left") -> str:
    padding = " " * max(0, width - display_width(text))
    return padding + str(text) if align == "right" else str(text) + padding


def emit_warning(report: Optional[RunReport], message: str, *args: Any) -> None:
    """警告を logger と RunReport の両方に流す。report が無い場合でも
    最低限ログには残す。"""
    text = message % args if args else message
    if report is not None:
        report.warn(text)
    else:
        logger.warning(text)

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [3/8] 設定
# ------------------------------------------------------------
# 右側のフォームで指定する値はこのセルに集約している。
# 実行前に触るのは基本ここだけ。
# ============================================================

# --- 対象試合とチーム -----------------------------------------
TARGET_MATCH_URL = "https://www.football-lab.jp/nago/report?year=2026&month=09&date=06"  # @param {type:"string"}
MY_TEAM = "名古屋グランパス"  # @param {type:"string"}
NEXT_TEAM = "V・ファーレン長崎"  # @param {type:"string"}
MY_TEAM_COLOR = "#d80c18"  # @param {type:"string"}
OPPONENT_TEAM_COLOR = "#005ce6"  # @param {type:"string"}

# --- CBPランキングのカテゴリ -----------------------------------
# Ver.8 までは j1001 固定だったため、J2/J3 のクラブを指定すると
# J1のリーグ平均で割った相対値が出ていた。auto はランキング表に
# 自チームが載っているカテゴリを実測で判定する。
CBP_RANKING_CATEGORY = "auto"  # @param ["auto", "j1", "j2", "j3"]
CBP_RANKING_YEAR = "100"  # @param {type:"string"}

# --- Drive 一時画像の扱い --------------------------------------
# Docs に画像を貼るには一度 Drive に公開リンクで置く必要があるが、
# 貼り付け後は公開したままにしない。
DRIVE_TEMP_FOLDER_NAME = "Konwenga Helper 一時画像"  # @param {type:"string"}
REVOKE_PUBLIC_LINKS_AFTER_EXPORT = True  # @param {type:"boolean"}
DRIVE_TEMP_RETENTION_DAYS = 7  # @param {type:"integer"}

# --- ログ ------------------------------------------------------
LOG_LEVEL = "INFO"  # @param ["INFO", "DEBUG", "WARNING"]

setup_logging(LOG_LEVEL)


# ------------------------------------------------------------
# 設定オブジェクト
# ------------------------------------------------------------
class ScraperSelectors:
    """Football LAB の DOM 依存を1か所に集約する。
    相手サイトの構造が変わったときに見る場所をここだけにする。"""

    USER_AGENT: Final[str] = "Konwenga-Helper/9.0 (+https://grapo.net/)"
    H3_BOX_HEADER: Final[str] = "h3.boxHeader"
    H3_BOX_HEADER_SPAN: Final[str] = "h3.boxHeader span"
    TBL_COMPARE: Final[str] = "div.tblCompare"
    TBL_COMPARE_CLASS: Final[str] = "tblCompare"
    STATS_TABLE_CLASS: Final[str] = "statsTbl6"
    TEAM_NAME_SP: Final[str] = "span.sp"
    TEAM_NAME_DSKTP: Final[str] = "span.dsktp"
    TEAM_TITLE: Final[str] = "h4.tName"
    BOX_HALF_CLASS: Final[str] = "boxHalf"
    STATS_TABLE: Final[str] = "table.statsTbl6"
    TAB_BOX_CBP: Final[str] = 'div.tabBox[data-name="cbp"]'
    BOX_TIMELINE: Final[str] = "div.boxTimeline"
    TIMEBASE_CLASS: Final[str] = "timebase"
    HOME_POSS_CLASS: Final[str] = "homePoss"
    AWAY_POSS_CLASS: Final[str] = "awayPoss"
    DATA_TTL_CLASS: Final[str] = "dataTtl"
    HOME_SHOT_CLASS: Final[str] = "homeShot"
    AWAY_SHOT_CLASS: Final[str] = "awayShot"
    CHART_HOME_DIV: Final[str] = "#chartHomeShots"
    CHART_AWAY_DIV: Final[str] = "#chartAwayShots"
    CHART_ANY_DIV: Final[str] = 'div[id*="Shots"]'
    CHART_ANY_SVG: Final[str] = "#chartHomeShots svg, #chartAwayShots svg, div[id*='Shots'] svg"
    SHOT_BACKGROUND_PATH: Final[str] = "/img/graph/shot.png"
    CBP_RANKING_TABLE_ID: Final[str] = "ls_teamCBP"


# 互換のため旧名も残す
ScraperConfig = ScraperSelectors


class Kit(str, Enum):
    HOME = "HOME"
    AWAY = "AWAY"


class CbpCategory(str, Enum):
    """Football LAB のCBPランキングはカテゴリごとに別URL。"""

    J1 = "j1"
    J2 = "j2"
    J3 = "j3"

    @property
    def ranking_code(self) -> str:
        return {"j1": "j1001", "j2": "j2001", "j3": "j3001"}[self.value]

    @property
    def display_name(self) -> str:
        return self.value.upper()


CBP_CATEGORY_AUTO: Final[str] = "auto"


@dataclass(frozen=True)
class NetworkConfig:
    timeout_sec: int = 30
    cbp_timeout_sec: int = 15
    shot_timeout_sec: int = 60
    max_workers: int = 4
    user_agent: str = ScraperSelectors.USER_AGENT
    default_encoding: str = "utf-8"


@dataclass(frozen=True)
class CbpConfig:
    """CBPランキングの取得条件。category が None のとき auto 判定。"""

    category: Optional[CbpCategory] = None
    year: str = "100"
    auto_detect_order: Tuple[CbpCategory, ...] = (CbpCategory.J1, CbpCategory.J2, CbpCategory.J3)


@dataclass(frozen=True)
class DriveConfig:
    temp_folder_name: str = "Konwenga Helper 一時画像"
    revoke_public_links_after_export: bool = True
    retention_days: int = 7


@dataclass(frozen=True)
class TimelineStyleConfig:
    bar_width: float = 0.8
    goal_fontsize: int = 14
    stack_step: float = 6.0
    stack_margin: float = 4.0
    goal_pedestal_scale: float = 1.5
    timeline_figsize: Tuple[int, int] = (10, 5)
    timeline_dpi: int = 150


@dataclass(frozen=True)
class RadarStyleConfig:
    radar_figsize: Tuple[int, int] = (8, 8)
    radar_dpi: int = 150

    # タイトル
    title_y: float = 1.08
    title_fontsize: int = 15

    # 線・塗り
    team_line_width: float = 2.0
    league_line_width: float = 2.0
    team_fill_alpha: float = 0.15
    league_linestyle: str = "dashed"

    # 凡例
    legend_loc: str = "lower center"
    legend_bbox_x: float = 0.5
    legend_bbox_y: float = -0.18
    legend_ncol: int = 3
    legend_frameon: bool = False
    legend_borderaxespad: float = 0.0

    # レーダーチャート本体の配置領域
    # [left, bottom, width, height] を 0.0～1.0 のFigure座標で指定
    axes_left: float = 0.16
    axes_bottom: float = 0.24
    axes_width: float = 0.68
    axes_height: float = 0.62

    # 半径ラベル位置
    rlabel_position: float = 90

    # 保存（tight で非対称に切り取られるのを避ける）
    savefig_bbox_inches: Optional[str] = None
    savefig_pad_inches: float = 0.10


@dataclass(frozen=True)
class DocsStyleConfig:
    radar_image_size_pt: Tuple[int, int] = (350, 350)
    timeline_image_size_pt: Tuple[int, int] = (500, 350)
    shot_image_size_pt: Tuple[int, int] = (300, 300)
    google_api_max_retries: int = 6
    google_api_initial_wait_sec: float = 1.0
    google_api_max_wait_sec: float = 16.0
    google_api_request_interval_sec: float = 0.05

    # 表のレイアウト
    table_font_pt: float = 11.0
    table_cell_padding_pt: float = 12.0
    table_min_column_width_pt: float = 26.0
    table_max_total_width_pt: float = 450.0
    table_align_center: bool = True

    # 表のチーム名セル
    my_team_cell_bg: str = "#d80c18"
    my_team_cell_text: str = "#ffffff"
    opponent_team_cell_bg: str = "#cfe8fa"


@dataclass(frozen=True)
class AppConfig:
    target_match_url: str
    my_team: str
    next_team: str
    my_team_color: str
    opponent_team_color: str
    network: NetworkConfig = field(default_factory=NetworkConfig)
    cbp: CbpConfig = field(default_factory=CbpConfig)
    drive: DriveConfig = field(default_factory=DriveConfig)
    timeline_style: TimelineStyleConfig = field(default_factory=TimelineStyleConfig)
    radar_style: RadarStyleConfig = field(default_factory=RadarStyleConfig)
    docs_style: DocsStyleConfig = field(default_factory=DocsStyleConfig)


# ------------------------------------------------------------
# 色まわりのユーティリティ
# ------------------------------------------------------------
DEFAULT_MY_TEAM_COLOR: Final[str] = "#d80c18"
DEFAULT_OPPONENT_TEAM_COLOR: Final[str] = "#005ce6"


def normalize_hex_color(color: str) -> str:
    text = (color or "").strip().lower()
    if not text:
        return "#999999"
    return text if text.startswith("#") else "#" + text


def validate_hex_color(color: str, fallback: str) -> str:
    normalized = normalize_hex_color(color)
    if re.fullmatch(r"#[0-9a-fA-F]{6}", normalized):
        return normalized.lower()
    logger.warning("色指定 %r は #RRGGBB 形式ではないため %s を使用します", color, fallback)
    return fallback.lower()


# 旧名（既存コードからの呼び出し互換）
_norm_hex = normalize_hex_color
_validate_hex_color = validate_hex_color


def resolve_cbp_category(value: Union[str, CbpCategory, None]) -> Optional[CbpCategory]:
    """フォームの文字列を CbpCategory に変換する。auto / 空文字は None。"""
    if value is None:
        return None
    if isinstance(value, CbpCategory):
        return value
    text = str(value).strip().lower()
    if not text or text == CBP_CATEGORY_AUTO:
        return None
    try:
        return CbpCategory(text)
    except ValueError:
        logger.warning("CBPカテゴリ %r は認識できないため auto 判定に切り替えます", value)
        return None


def build_app_config(
    *,
    target_match_url: str,
    my_team: str,
    next_team: str,
    my_team_color: str = DEFAULT_MY_TEAM_COLOR,
    opponent_team_color: str = DEFAULT_OPPONENT_TEAM_COLOR,
    cbp_category: Union[str, CbpCategory, None] = CBP_CATEGORY_AUTO,
    cbp_year: str = "100",
    drive_temp_folder_name: str = "Konwenga Helper 一時画像",
    revoke_public_links_after_export: bool = True,
    drive_retention_days: int = 7,
) -> AppConfig:
    """Ver.8 では main() が戻り値を組み直して色を検証していたが、
    検証はここで完結させる。グローバル変数も参照しない純関数。"""
    return AppConfig(
        target_match_url=(target_match_url or "").strip(),
        my_team=(my_team or "").strip(),
        next_team=(next_team or "").strip(),
        my_team_color=validate_hex_color(my_team_color, DEFAULT_MY_TEAM_COLOR),
        opponent_team_color=validate_hex_color(opponent_team_color, DEFAULT_OPPONENT_TEAM_COLOR),
        cbp=CbpConfig(category=resolve_cbp_category(cbp_category), year=str(cbp_year or "100").strip()),
        drive=DriveConfig(
            temp_folder_name=(drive_temp_folder_name or "Konwenga Helper 一時画像").strip(),
            revoke_public_links_after_export=bool(revoke_public_links_after_export),
            retention_days=int(drive_retention_days or 0),
        ),
    )


def build_session(network: NetworkConfig) -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=max(4, network.max_workers),
        pool_maxsize=max(8, network.max_workers * 2),
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": network.user_agent})
    return session


def decode_response_text(response: requests.Response, network: NetworkConfig) -> str:
    """apparent_encoding はレスポンス全文を chardet で走査するため重い。
    Football LAB は UTF-8 固定なので既定値で読む。"""
    if not response.encoding:
        response.encoding = network.default_encoding
    return response.text

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [4/8] クラブマスタ（色・別名）
# ------------------------------------------------------------
# 昇降格のたびに触るのはこのセルのテーブルだけ。
# ページのチームセレクタから別名を学習する仕組みがあるので、
# 未登録クラブがあっても実行自体は止まらない（警告で知らせる）。
# ============================================================
RAW_CLUB_COLORS = {
  # J1 (2026/27)
  "KASHIMA": {"home": "#b71840", "away": "#ffffff"}, "MITO": {"home": "#1d1f90", "away": "#1d1f90"},
  "URAWA": {"home": "#e7002b", "away": "#ffffff"}, "CHIBA": {"home": "#fee100", "away": "#fee100"},
  "KASHIWA": {"home": "#fff100", "away": "#fff100"}, "TOKYO": {"home": "#214198", "away": "#ffffff"},
  "VERDY": {"home": "#03764b", "away": "#03764b"}, "MACHIDA": {"home": "#00236a", "away": "#00236a"},
  "KAWASAKI": {"home": "#35a0d9", "away": "#35a0d9"}, "MARINOS": {"home": "#003989", "away": "#003989"},
  "SHIMIZU": {"home": "#faa528", "away": "#faa528"}, "NAGOYA": {"home": "#d80c18", "away": "#ffffff"},
  "SANGA": {"home": "#74006b", "away": "#74006b"}, "GAMBA": {"home": "#093fa6", "away": "#093fa6"},
  "CEREZO": {"home": "#d40069", "away": "#ffffff"}, "KOBE": {"home": "#8f0a1f", "away": "#ffffff"},
  "OKAYAMA": {"home": "#1d2468", "away": "#1d2468"}, "HIROSHIMA": {"home": "#50318f", "away": "#50318f"},
  "FUKUOKA": {"home": "#04407f", "away": "#04407f"}, "NAGASAKI": {"home": "#0d4fc7", "away": "#0d4fc7"},
  # J2 (2026)
  "CONSA": {"home": "#d7000f", "away": "#ffffff"}, "HACHINOHE": {"home": "#00b0e8", "away": "#00b0e8"},
  "SENDAI": {"home": "#fccc00", "away": "#fccc00"}, "AKITA": {"home": "#003f98", "away": "#003f98"},
  "YAMAGATA": {"home": "#1f60a8", "away": "#1f60a8"}, "IWAKI": {"home": "#d2232a", "away": "#d2232a"},
  "TOCHIGICITY": {"home": "#0b3f8c", "away": "#0b3f8c"}, "OMIYA": {"home": "#00407b", "away": "#00407b"},
  "YOKOHAMAFC": {"home": "#00a0e4", "away": "#00a0e4"}, "SHONAN": {"home": "#67b464", "away": "#67b464"},
  "KOFU": {"home": "#0a2c82", "away": "#0a2c82"}, "NIIGATA": {"home": "#ff6600", "away": "#ff6600"},
  "TOYAMA": {"home": "#0d55a5", "away": "#0d55a5"}, "IWATA": {"home": "#6e9dd3", "away": "#6e9dd3"},
  "FUJIEDA": {"home": "#8a4c9c", "away": "#8a4c9c"}, "TOKUSHIMA": {"home": "#111183", "away": "#111183"},
  "IMABARI": {"home": "#0068b7", "away": "#0068b7"}, "TOSU": {"home": "#0096d2", "away": "#0096d2"},
  "OITA": {"home": "#0b58a5", "away": "#0b58a5"}, "MIYAZAKI": {"home": "#0a9a4a", "away": "#0a9a4a"},
  # J3 (2026)
  "FUKUSHIMA": {"home": "#c8102e", "away": "#c8102e"}, "TOCHIGISC": {"home": "#f6ad3c", "away": "#f6ad3c"},
  "GUNMA": {"home": "#0b3d91", "away": "#0b3d91"}, "SAGAMIHARA": {"home": "#00a0c6", "away": "#00a0c6"},
  "MATSUMOTO": {"home": "#12603a", "away": "#12603a"}, "NAGANO": {"home": "#f39800", "away": "#f39800"},
  "KANAZAWA": {"home": "#e50009", "away": "#ffffff"}, "GIFU": {"home": "#004017", "away": "#004017"},
  "SHIGA": {"home": "#005bab", "away": "#005bab"}, "FCOSAKA": {"home": "#00478a", "away": "#00478a"},
  "NARA": {"home": "#0f4c9e", "away": "#0f4c9e"}, "TOTTORI": {"home": "#1b9c4a", "away": "#1b9c4a"},
  "YAMAGUCHI": {"home": "#e8531f", "away": "#e8531f"}, "SANUKI": {"home": "#0075c2", "away": "#0075c2"},
  "EHIME": {"home": "#f39700", "away": "#f39700"}, "KOCHI": {"home": "#e60012", "away": "#e60012"},
  "KITAKYUSHU": {"home": "#ffe100", "away": "#ffe100"}, "KUMAMOTO": {"home": "#c8102e", "away": "#c8102e"},
  "KAGOSHIMA": {"home": "#0b2c6a", "away": "#0b2c6a"}, "RYUKYU": {"home": "#e60012", "away": "#e60012"}
}

# 正式名称・略称・愛称を登録する。_norm_key が NFKC 正規化するため、全角と半角の
# 両方を書く必要はない（"ＦＣ東京" と "FC東京" は同じキーになる）。
RAW_TEAM_ALIASES = {
  # --- J1 (2026/27) ---
  "KASHIMA": ["KASHIMA", "KASM", "鹿島アントラーズ", "アントラーズ", "鹿島"],
  "MITO": ["MITO", "水戸ホーリーホック", "ホーリーホック", "水戸"],
  "URAWA": ["URAWA", "URAW", "浦和レッズ", "レッズ", "浦和"],
  "CHIBA": ["CHIBA", "CHIB", "ジェフユナイテッド千葉", "ジェフユナイテッド市原・千葉", "ジェフ千葉", "ジェフ", "千葉"],
  "KASHIWA": ["KASHIWA", "KASW", "柏レイソル", "レイソル", "柏"],
  "TOKYO": ["TOKYO", "FCTK", "FC東京", "東京"],
  "VERDY": ["VERDY", "TK-V", "東京ヴェルディ", "ヴェルディ", "東京V"],
  "MACHIDA": ["MACHIDA", "MCD", "FC町田ゼルビア", "町田ゼルビア", "ゼルビア", "町田"],
  "KAWASAKI": ["KAWASAKI", "KA-F", "川崎フロンターレ", "フロンターレ", "川崎F", "川崎"],
  "MARINOS": ["MARINOS", "Y-FM", "横浜F・マリノス", "横浜Fマリノス", "Fマリノス", "マリノス", "横浜FM"],
  "SHIMIZU": ["SHIMIZU", "SHIM", "清水エスパルス", "エスパルス", "清水"],
  "NAGOYA": ["NAGOYA", "NAGO", "名古屋グランパス", "グランパス", "名古屋"],
  "SANGA": ["SANGA", "KYOT", "京都サンガF.C.", "京都サンガFC", "京都サンガ", "サンガ", "京都"],
  "GAMBA": ["GAMBA", "G-OS", "ガンバ大阪", "ガンバ", "G大阪"],
  "CEREZO": ["CEREZO", "C-OS", "セレッソ大阪", "セレッソ", "C大阪"],
  "KOBE": ["KOBE", "ヴィッセル神戸", "ヴィッセル", "神戸"],
  "OKAYAMA": ["OKAYAMA", "OKAY", "ファジアーノ岡山", "ファジアーノ", "岡山"],
  "HIROSHIMA": ["HIROSHIMA", "HIRO", "サンフレッチェ広島", "サンフレッチェ", "広島"],
  "FUKUOKA": ["FUKUOKA", "FUKU", "アビスパ福岡", "アビスパ", "福岡"],
  "NAGASAKI": ["NAGASAKI", "NGSK", "V・ファーレン長崎", "Vファーレン長崎", "V・ファーレン", "Vファーレン", "長崎"],
  # --- J2 (2026) ---
  "CONSA": ["CONSA", "SAPP", "北海道コンサドーレ札幌", "コンサドーレ札幌", "コンサドーレ", "札幌"],
  "HACHINOHE": ["HACHINOHE", "HACH", "ヴァンラーレ八戸", "ヴァンラーレ", "八戸"],
  "SENDAI": ["SENDAI", "SEND", "ベガルタ仙台", "ベガルタ", "仙台"],
  "AKITA": ["AKITA", "AKI", "ブラウブリッツ秋田", "ブラウブリッツ", "秋田"],
  "YAMAGATA": ["YAMAGATA", "YAMA", "モンテディオ山形", "モンテディオ", "山形"],
  "IWAKI": ["IWAKI", "IFC", "いわきFC", "いわき"],
  "TOCHIGICITY": ["TOCHIGICITY", "TO-C", "栃木シティ", "栃木C"],
  "OMIYA": ["OMIYA", "OMIY", "RB大宮アルディージャ", "大宮アルディージャ", "アルディージャ", "RB大宮", "大宮"],
  "YOKOHAMAFC": ["YOKOHAMAFC", "Y-FC", "横浜FC"],
  "SHONAN": ["SHONAN", "SHON", "湘南ベルマーレ", "ベルマーレ", "湘南"],
  "KOFU": ["KOFU", "ヴァンフォーレ甲府", "ヴァンフォーレ", "甲府"],
  "NIIGATA": ["NIIGATA", "NIIG", "アルビレックス新潟", "アルビレックス", "新潟"],
  "TOYAMA": ["TOYAMA", "TOYA", "カターレ富山", "カターレ", "富山"],
  "IWATA": ["IWATA", "IWAT", "ジュビロ磐田", "ジュビロ", "磐田"],
  "FUJIEDA": ["FUJIEDA", "FUJI", "藤枝MYFC", "藤枝"],
  "TOKUSHIMA": ["TOKUSHIMA", "TOKU", "徳島ヴォルティス", "ヴォルティス", "徳島"],
  "IMABARI": ["IMABARI", "IMAB", "FC今治", "今治"],
  "TOSU": ["TOSU", "サガン鳥栖", "サガン", "鳥栖"],
  "OITA": ["OITA", "大分トリニータ", "トリニータ", "大分"],
  "MIYAZAKI": ["MIYAZAKI", "MYZK", "テゲバジャーロ宮崎", "テゲバジャーロ", "宮崎"],
  # --- J3 (2026) ---
  "FUKUSHIMA": ["FUKUSHIMA", "FKSM", "福島ユナイテッドFC", "福島ユナイテッド", "福島"],
  "TOCHIGISC": ["TOCHIGISC", "TO-S", "栃木SC"],
  "GUNMA": ["GUNMA", "GNM", "ザスパ群馬", "ザスパ", "群馬"],
  "SAGAMIHARA": ["SAGAMIHARA", "SAGM", "SC相模原", "相模原"],
  "MATSUMOTO": ["MATSUMOTO", "MATS", "松本山雅FC", "松本山雅", "松本"],
  "NAGANO": ["NAGANO", "NAGA", "AC長野パルセイロ", "長野パルセイロ", "パルセイロ", "長野"],
  "KANAZAWA": ["KANAZAWA", "KANA", "ツエーゲン金沢", "ツェーゲン金沢", "ツエーゲン", "金沢"],
  "GIFU": ["GIFU", "FC岐阜", "岐阜"],
  "SHIGA": ["SHIGA", "RSFC", "レイラック滋賀FC", "レイラック滋賀", "レイラック", "滋賀"],
  "FCOSAKA": ["FCOSAKA", "F-OS", "FC大阪"],
  "NARA": ["NARA", "奈良クラブ", "奈良"],
  "TOTTORI": ["TOTTORI", "TOTR", "ガイナーレ鳥取", "ガイナーレ", "鳥取"],
  "YAMAGUCHI": ["YAMAGUCHI", "R-YA", "レノファ山口FC", "レノファ山口", "レノファ", "山口"],
  "SANUKI": ["SANUKI", "SANU", "カマタマーレ讃岐", "カマタマーレ", "讃岐"],
  "EHIME": ["EHIME", "EHIM", "愛媛FC", "愛媛"],
  "KOCHI": ["KOCHI", "KUSC", "高知ユナイテッドSC", "高知ユナイテッド", "高知"],
  "KITAKYUSHU": ["KITAKYUSHU", "KIKY", "ギラヴァンツ北九州", "ギラヴァンツ", "北九州"],
  "KUMAMOTO": ["KUMAMOTO", "KUMA", "ロアッソ熊本", "ロアッソ", "熊本"],
  "KAGOSHIMA": ["KAGOSHIMA", "KUFC", "鹿児島ユナイテッドFC", "鹿児島ユナイテッド", "鹿児島"],
  "RYUKYU": ["RYUKYU", "RYUK", "FC琉球", "琉球"]
}


def normalize_team_key(name: str) -> str:
    """NFKC で全角と半角を揃える。Football LAB は正式名称を全角（ＦＣ、
    Ｆ・マリノス）で出す箇所があり、これが無いと半角で登録した別名と
    一致しない。"""
    text = unicodedata.normalize("NFKC", (name or "").strip()).casefold()
    for char in (" ", "　", "-", "‐", "–", "—", "_", ".", "・", "／", "/"):
        text = text.replace(char, "")
    return text


# 旧名（既存コードからの呼び出し互換）
_norm_key = normalize_team_key


@dataclass(frozen=True)
class ClubConfig:
    club_colors: Dict[str, Tuple[str, str]]
    team_alias_map: Dict[str, str]

    def resolve_team_abbr(self, team: str) -> str:
        return self.team_alias_map.get(normalize_team_key(team), (team or "").strip().upper())

    def is_known_team(self, team: str) -> bool:
        """別名テーブルに載っているチーム名かどうか。resolve_team_abbr は未知でも
        大文字化した入力をそのまま返すため、判定にはこちらを使う。"""
        return bool(team) and normalize_team_key(team) in self.team_alias_map

    def get_club_color(self, team: str, kit: Union[Kit, str], default: Optional[str] = None) -> str:
        abbr = self.resolve_team_abbr(team)
        resolved_kit = self._coerce_kit(kit, default)
        if resolved_kit is None:
            return normalize_hex_color(default or "#999999")
        colors = self.club_colors.get(abbr)
        if colors is None:
            return normalize_hex_color(default or "#999999")
        home_color, away_color = colors
        return home_color if resolved_kit is Kit.HOME else away_color

    @staticmethod
    def _coerce_kit(kit: Union[Kit, str], default: Optional[str]) -> Optional[Kit]:
        if isinstance(kit, Kit):
            return kit
        text = str(kit).strip().upper()
        if "." in text:
            text = text.split(".")[-1]
        try:
            return Kit(text)
        except ValueError:
            if default is None:
                raise ValueError(f"Invalid kit: {kit}")
            return None


@lru_cache(maxsize=1)
def load_club_config() -> ClubConfig:
    club_colors: Dict[str, Tuple[str, str]] = {}
    alias_map: Dict[str, str] = {}
    for abbr, colors in RAW_CLUB_COLORS.items():
        if isinstance(colors, dict):
            club_colors[str(abbr).strip().upper()] = (
                normalize_hex_color(str(colors.get("home", ""))),
                normalize_hex_color(str(colors.get("away", ""))),
            )
    for abbr, aliases in RAW_TEAM_ALIASES.items():
        canonical = str(abbr).strip().upper()
        alias_map[normalize_team_key(canonical)] = canonical
        for alias in aliases:
            if key := normalize_team_key(str(alias)):
                alias_map[key] = canonical
    return ClubConfig(club_colors=club_colors, team_alias_map=alias_map)


# Football LAB の全ページに、J1/J2/J3全クラブへのリンクを含むチームセレクタが埋まっている。
# <a href="/nago"><img alt="名古屋グランパス" src=".../NAGO_sdw.png">名古屋</a> という形なので、
# ここから「正式名称」「略称」「チームコード」を毎回拾えば、昇降格でクラブが入れ替わっても
# 別名テーブルを手で直さずに済む。
TEAM_NAV_HREF_PATTERN = re.compile(r"^(?:https?://[^/]+)?/([a-z0-9\-]+)/?$")
TEAM_NAV_IMG_PATTERN = re.compile(r"/img/team/([A-Za-z0-9\-]+)_", re.IGNORECASE)


def extract_team_directory(soup: BeautifulSoup) -> Dict[str, Dict[str, str]]:
    """ページ内のチームセレクタから {チームコード: {"official": 正式名称, "short": 略称}} を作る。"""
    directory: Dict[str, Dict[str, str]] = {}
    for anchor in soup.find_all("a", href=True):
        image = anchor.find("img")
        if image is None:
            continue
        source = str(image.get("src") or "")
        if not (image_match := TEAM_NAV_IMG_PATTERN.search(source)):
            continue
        if not (href_match := TEAM_NAV_HREF_PATTERN.match(str(anchor.get("href")).strip())):
            continue
        code = image_match.group(1).strip().upper()
        official_name = str(image.get("alt") or "").strip()
        short_name = anchor.get_text(strip=True)
        # アンカーテキストは alt と連結されることがあるので、正式名称ぶんを取り除く
        if official_name and short_name.startswith(official_name):
            short_name = short_name[len(official_name):].strip()
        if not official_name and not short_name:
            continue
        entry = directory.setdefault(code, {"official": "", "short": "", "slug": href_match.group(1)})
        if official_name and not entry["official"]:
            entry["official"] = official_name
        if short_name and not entry["short"]:
            entry["short"] = short_name
    return directory


@dataclass(frozen=True)
class ClubAliasReport:
    found: int = 0
    learned: List[str] = field(default_factory=list)
    added_aliases: List[str] = field(default_factory=list)
    missing_colors: List[str] = field(default_factory=list)


def augment_club_config_from_html(
    club_config: ClubConfig,
    soup: Optional[BeautifulSoup],
) -> Tuple[ClubConfig, ClubAliasReport]:
    """静的テーブルを土台に、ページから読んだクラブ名で別名を補う。
    既存の別名が優先されるので、手で入れた色や略称の割り当ては壊れない。"""
    if soup is None:
        return club_config, ClubAliasReport()

    directory = extract_team_directory(soup)
    if not directory:
        return club_config, ClubAliasReport()

    alias_map = dict(club_config.team_alias_map)
    newly_coined_abbrs: set = set()
    learned: List[str] = []
    added_aliases: List[str] = []
    for code, entry in sorted(directory.items()):
        names = [name for name in (entry.get("official"), entry.get("short"), code) if name]
        # 既知の名前が1つでもあれば、そのクラブの略号に寄せる
        abbr = next((alias_map[normalize_team_key(name)] for name in names if normalize_team_key(name) in alias_map), "")
        if not abbr:
            abbr = code.replace("-", "")
            newly_coined_abbrs.add(abbr)
            learned.append(f"{entry.get('official') or entry.get('short') or code}({code}->{abbr})")
        added = [name for name in names if normalize_team_key(name) and normalize_team_key(name) not in alias_map]
        for name in added:
            alias_map[normalize_team_key(name)] = abbr
        if added and abbr not in newly_coined_abbrs:
            added_aliases.append(f"{abbr}: {'/'.join(added)}")

    augmented = ClubConfig(club_colors=club_config.club_colors, team_alias_map=alias_map)
    missing_colors = [abbr for abbr in sorted(set(alias_map.values())) if abbr not in augmented.club_colors]
    report = ClubAliasReport(
        found=len(directory),
        learned=learned,
        added_aliases=added_aliases,
        missing_colors=missing_colors,
    )
    return augmented, report


def log_club_config(club_config: ClubConfig, alias_report: ClubAliasReport, report: Optional[RunReport] = None) -> None:
    abbrs = sorted(set(club_config.team_alias_map.values()))
    lines = [
        "",
        "--- クラブ別名テーブル確認 ---",
        f"登録クラブ数: {len(abbrs)} / 別名キー数: {len(club_config.team_alias_map)}",
        f"ページのチームセレクタから検出したクラブ数: {alias_report.found}",
    ]
    if alias_report.added_aliases:
        lines.append(f"既存クラブに別名を追加 ({len(alias_report.added_aliases)}件): {alias_report.added_aliases}")
    if not alias_report.learned and not alias_report.added_aliases:
        lines.append("静的テーブルだけで全クラブを解決できています")
    lines.append("----------------------------")
    logger.info("\n".join(lines))

    # Ver.8 では print で流していたため見落としやすかった。警告として集約する。
    if alias_report.learned:
        emit_warning(
            report,
            "別名テーブル未登録のクラブを自動採番しました（色は既定色になります）。RAW_CLUB_COLORS / RAW_TEAM_ALIASES への追記を推奨します: %s",
            alias_report.learned,
        )
    if alias_report.missing_colors:
        emit_warning(report, "色が未登録の略号があります: %s", alias_report.missing_colors)
    if alias_report.found == 0:
        emit_warning(report, "ページからチームセレクタを検出できませんでした。Football LAB の構造が変わった可能性があります")

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [5/8] スクレイピング
# ------------------------------------------------------------
# Ver.9 では、表の列数が想定と違う場合に黙って生の DataFrame を
# 返していた箇所をすべて警告として拾い上げる。相手サイトの構造が
# 変わったら「崩れた表がそのまま Docs に載る」のではなく、実行結果
# サマリに残るようにした。
# ============================================================


@dataclass(frozen=True)
class TeamSvg:
    team_name: str
    svg_text: str


class FootballLabScraper:
    #  read_html が返す列数の想定値。ここを外れたら構造変更の疑いあり。
    CBP_SUMMARY_COLUMNS: Final[int] = 7
    STATS_COLUMNS: Final[int] = 7
    CBP_DETAIL_MIN_COLUMNS: Final[int] = 5
    CBP_DETAIL_SECTION_TITLES: Final[Tuple[str, ...]] = (
        "1. 攻撃", "2. パス", "3. クロス", "4. ドリブル",
        "5. シュート", "6. 奪取", "7. 守備", "8. セーブ",
    )

    def __init__(
        self,
        url: str,
        session: requests.Session,
        network_config: Optional[NetworkConfig] = None,
        report: Optional[RunReport] = None,
    ):
        self.url = url
        self.session = session
        self.network_config = network_config or NetworkConfig()
        self.report = report
        self.soup: Optional[BeautifulSoup] = None
        self.html_text: str = ""
        self.page_title: str = ""
        self.file_date: str = self._parse_url_date()

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    # ---- 取得 ------------------------------------------------
    def _parse_url_date(self) -> str:
        """Ver.8 では例外時に "unknown_date" が残り、ファイル名と Docs
        タイトルの両方が unknown_date になっていた。今日の日付に落とす。"""
        fallback = time.strftime("%Y%m%d")
        try:
            params = parse_qs(urlparse(self.url).query)
            year = params.get("year", [""])[0]
            month = params.get("month", [""])[0]
            day = params.get("date", [""])[0]
            if year and month and day:
                return f"{year}{str(month).zfill(2)}{str(day).zfill(2)}"
            logger.warning("URLから試合日を取得できないため実行日 %s を使用します: %s", fallback, self.url)
            return fallback
        except Exception:
            logger.exception("URL日付解析に失敗したため実行日 %s を使用します", fallback)
            return fallback

    def fetch(self, timeout_sec: Optional[int] = None) -> bool:
        if not self.url:
            logger.error("対象URLが空です")
            return False
        try:
            response = self.session.get(self.url, timeout=timeout_sec or self.network_config.timeout_sec)
            response.raise_for_status()
            self.html_text = decode_response_text(response, self.network_config)
            self.soup = BeautifulSoup(self.html_text, "html.parser")
            self.page_title = self.soup.title.get_text(strip=True) if self.soup.title else "Match Report"
            return True
        except requests.HTTPError as error:
            status = getattr(getattr(error, "response", None), "status_code", "不明")
            logger.error("試合レポート取得に失敗しました (HTTP %s): %s", status, self.url)
            logger.error("URLの年月日と試合の有無を確認してください。試合がない日付だと404になります。")
            return False
        except requests.RequestException:
            logger.exception("試合レポート取得に失敗しました（接続またはタイムアウト）: %s", self.url)
            return False
        except Exception:
            logger.exception("試合レポート解析に失敗しました: %s", self.url)
            return False

    # ---- 構造化 ----------------------------------------------
    def get_structure(self) -> List[Dict[str, Any]]:
        if not self.soup:
            self._warn("試合レポートのHTMLが未取得のため、Docs用の構造を作れませんでした")
            return []
        structure: List[Dict[str, Any]] = [{"type": "title", "content": self.page_title}]
        structure.extend(self._process_section("チャンスビルディングポイント", "cbp_summary"))
        structure.extend(self._get_cbp_details())
        structure.extend(self._process_section("スタッツ", "stats"))
        table_count = sum(1 for item in structure if item.get("type") == "table")
        if table_count == 0:
            self._warn("試合レポートから表を1つも抽出できませんでした。Football LAB の構造が変わった可能性があります")
        logger.info("試合レポートから %s 個の表を抽出しました", table_count)
        return structure

    def _get_team_abbreviations(self, tbl_compare_div) -> Tuple[str, str]:
        home_abbr, away_abbr = "HOME", "AWAY"
        if not tbl_compare_div:
            self._warn("チーム比較表(div.tblCompare)が見つからないため、チーム略称を HOME/AWAY で代用します")
            return home_abbr, away_abbr
        em_tags = tbl_compare_div.find_all("em")
        try:
            if len(em_tags) > 1 and (span := em_tags[1].select_one(ScraperSelectors.TEAM_NAME_SP)):
                home_abbr = span.get_text(strip=True) or home_abbr
            if len(em_tags) > 4 and (span := em_tags[4].select_one(ScraperSelectors.TEAM_NAME_SP)):
                away_abbr = span.get_text(strip=True) or away_abbr
        except AttributeError:
            logger.exception("チーム略称の抽出に失敗しました")
        if home_abbr == "HOME" or away_abbr == "AWAY":
            self._warn(
                "チーム略称を抽出できませんでした（home=%s, away=%s）。表の見出しが HOME/AWAY のまま出力されます",
                home_abbr, away_abbr,
            )
        return home_abbr, away_abbr

    def _process_section(self, keyword: str, section_type: str) -> List[Dict[str, Any]]:
        items: List[Dict[str, Any]] = []
        if not self.soup:
            return items
        target_heading = next(
            (
                heading
                for heading in self.soup.select(ScraperSelectors.H3_BOX_HEADER)
                if (span := heading.select_one("span")) and keyword in span.get_text()
            ),
            None,
        )
        if not target_heading:
            self._warn("見出し「%s」が見つかりませんでした。該当セクションは出力されません", keyword)
            return items
        items.append({"type": "header", "content": keyword})

        node = target_heading.next_sibling
        compare_div = None
        stats_table = None
        while node:
            node_name = getattr(node, "name", None)
            node_classes = node.get("class") or [] if node_name else []
            if node_name == "div" and ScraperSelectors.TBL_COMPARE_CLASS in node_classes:
                compare_div = node
            elif node_name == "table" and ScraperSelectors.STATS_TABLE_CLASS in node_classes:
                stats_table = node
                break
            node = node.next_sibling

        home_abbr, away_abbr = self._get_team_abbreviations(compare_div)
        if not stats_table:
            self._warn("見出し「%s」の直後にデータ表(table.statsTbl6)が見つかりませんでした", keyword)
            return items

        try:
            frames = pd.read_html(io.StringIO(str(stats_table)))
        except ValueError:
            logger.exception("HTML表の解析に失敗しました: section=%s", section_type)
            self._warn("「%s」の表を解析できませんでした", keyword)
            return items
        except Exception:
            logger.exception("セクション処理に失敗しました: section=%s", section_type)
            self._warn("「%s」の処理中に想定外のエラーが発生しました", keyword)
            return items

        if not frames:
            self._warn("「%s」の表が空でした", keyword)
            return items

        if section_type == "cbp_summary":
            formatted = self._format_cbp_summary(frames[0], home_abbr, away_abbr)
        elif section_type == "stats":
            formatted = self._format_stats(frames[0], home_abbr, away_abbr)
        else:
            formatted = frames[0].fillna("")
        items.append({"type": "table", "content": formatted})
        return items

    def _get_cbp_details(self) -> List[Dict[str, Any]]:
        items: List[Dict[str, Any]] = []
        if not self.soup:
            return items
        tab_box = self.soup.select_one(ScraperSelectors.TAB_BOX_CBP)
        if not tab_box:
            self._warn("CBP詳細のタブ(div.tabBox[data-name=\"cbp\"])が見つかりませんでした")
            return items

        home_abbr, away_abbr = self._get_team_abbreviations(self.soup.select_one(ScraperSelectors.TBL_COMPARE))
        items.append({"type": "header", "content": "チャンスビルディングポイント詳細"})
        try:
            frames = pd.read_html(io.StringIO(str(tab_box)))
        except ValueError:
            logger.exception("CBP詳細テーブル解析に失敗しました")
            self._warn("CBP詳細の表を解析できませんでした")
            return items
        except Exception:
            logger.exception("CBP詳細処理に失敗しました")
            self._warn("CBP詳細の処理中に想定外のエラーが発生しました")
            return items

        expected = len(self.CBP_DETAIL_SECTION_TITLES)
        if len(frames) < expected:
            self._warn("CBP詳細の表が %s 個しかありません（想定 %s 個）", len(frames), expected)
        for index, frame in enumerate(frames[:expected]):
            items.append({"type": "sub_header", "content": self.CBP_DETAIL_SECTION_TITLES[index]})
            items.append({
                "type": "table",
                "content": self._format_cbp_detail(frame, home_abbr, away_abbr, self.CBP_DETAIL_SECTION_TITLES[index]),
            })
        return items

    # ---- 整形（想定外の形状は警告する） --------------------------
    def _format_cbp_summary(self, frame: pd.DataFrame, home_abbr: str, away_abbr: str) -> pd.DataFrame:
        if len(frame.columns) != self.CBP_SUMMARY_COLUMNS:
            self._warn(
                "CBPサマリ表の列数が想定外です（実際 %s 列 / 想定 %s 列）。整形せずそのまま出力します",
                len(frame.columns), self.CBP_SUMMARY_COLUMNS,
            )
            return frame.fillna("")
        trimmed = frame.iloc[:, 1:6].copy()
        trimmed.columns = [
            f"[{home_abbr}] {trimmed.columns[0]}",
            f"[{home_abbr}] {trimmed.columns[1]}",
            "項目",
            f"[{away_abbr}] {str(trimmed.columns[3]).replace('.1', '')}",
            f"[{away_abbr}] {str(trimmed.columns[4]).replace('.1', '')}",
        ]
        return trimmed[trimmed.iloc[:, 0].astype(str) != trimmed["項目"].astype(str)].fillna("")

    def _format_stats(self, frame: pd.DataFrame, home_abbr: str, away_abbr: str) -> pd.DataFrame:
        if len(frame.columns) != self.STATS_COLUMNS:
            self._warn(
                "スタッツ表の列数が想定外です（実際 %s 列 / 想定 %s 列）。整形せずそのまま出力します",
                len(frame.columns), self.STATS_COLUMNS,
            )
            return frame.fillna("")
        frame = frame.copy()
        frame.columns = [
            f"[{home_abbr}] 平均", f"[{home_abbr}] 率", f"[{home_abbr}] 数",
            "項目",
            f"[{away_abbr}] 数", f"[{away_abbr}] 率", f"[{away_abbr}] 平均",
        ]
        frame = frame[frame.iloc[:, 0].astype(str) != frame.iloc[:, 3].astype(str)].copy()
        for column in (f"[{home_abbr}] 率", f"[{away_abbr}] 率"):
            if column in frame.columns:
                frame[column] = frame[column].astype(str).str.replace(r"[()]", "", regex=True)
        return frame.fillna("")

    def _format_cbp_detail(self, frame: pd.DataFrame, home_abbr: str, away_abbr: str, section_title: str = "") -> pd.DataFrame:
        if len(frame.columns) < self.CBP_DETAIL_MIN_COLUMNS:
            self._warn(
                "CBP詳細「%s」の列数が想定外です（実際 %s 列 / 最低 %s 列）。整形せずそのまま出力します",
                section_title or "(名称不明)", len(frame.columns), self.CBP_DETAIL_MIN_COLUMNS,
            )
            return frame.fillna("")
        trimmed = frame.drop(frame.columns[[1, 3]], axis=1)
        # 元表にヘッダ行が無く read_html が 0,1,2... を列名にするため、
        # 中央列の見出しがそのまま「2」として出力されていた。中身はCBPの値なので見出しは空にする。
        trimmed.columns = [home_abbr, "", away_abbr]
        return trimmed.fillna("")


def run_in_notebook(coro):
    """nest_asyncio.apply() 済みなので asyncio.run で足りる。
    get_event_loop() は Python 3.12 以降で非推奨。"""
    return asyncio.run(coro)


class FootballLabShotsSvgExporter:
    def __init__(
        self,
        target_url: str,
        session: requests.Session,
        network_config: Optional[NetworkConfig] = None,
        report: Optional[RunReport] = None,
    ):
        self.target_url = target_url
        self.session = session
        self.network_config = network_config or NetworkConfig()
        self.report = report
        self.background_image_url = urljoin(target_url, ScraperSelectors.SHOT_BACKGROUND_PATH)
        self.extraction_source: str = ""

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    def fetch_html(self, timeout_sec: Optional[int] = None) -> str:
        response = self.session.get(self.target_url, timeout=timeout_sec or self.network_config.timeout_sec)
        response.raise_for_status()
        return decode_response_text(response, self.network_config)

    def extract_from_static_html(self, html: str) -> List[TeamSvg]:
        soup = BeautifulSoup(html, "html.parser")
        found: List[TeamSvg] = []
        for selector in (ScraperSelectors.CHART_HOME_DIV, ScraperSelectors.CHART_AWAY_DIV):
            container = soup.select_one(selector)
            if not container:
                continue
            svg_node = container.find("svg")
            if not svg_node:
                continue
            box = container.find_parent("div", class_=ScraperSelectors.BOX_HALF_CLASS)
            found.append(TeamSvg(self._extract_team_name(box), self._normalize_svg(str(svg_node))))
        if len(found) >= 2:
            return found[:2]
        for container in soup.select(ScraperSelectors.CHART_ANY_DIV):
            if len(found) >= 2:
                break
            svg_node = container.find("svg")
            if not svg_node:
                continue
            box = container.find_parent("div", class_=ScraperSelectors.BOX_HALF_CLASS)
            found.append(TeamSvg(self._extract_team_name(box), self._normalize_svg(str(svg_node))))
        return found[:2]

    async def extract_via_playwright(self, timeout_sec: float = 30) -> List[TeamSvg]:
        found: List[TeamSvg] = []
        timeout_ms = int(timeout_sec * 1000)
        team_name_script = (
            "(el) => { const box = el.closest('div.boxHalf');"
            " const heading = box ? box.querySelector('h4.tName') : null;"
            " return heading ? heading.textContent.trim() : ''; }"
        )
        async with async_playwright() as playwright:
            browser = await playwright.chromium.launch(headless=True)
            try:
                page = await browser.new_page()
                try:
                    await page.goto(self.target_url, wait_until="domcontentloaded", timeout=timeout_ms)
                    await page.wait_for_selector(ScraperSelectors.CHART_ANY_SVG, timeout=timeout_ms)
                except PlaywrightTimeoutError:
                    logger.exception("PlaywrightでシュートチャートSVGの生成待ちがタイムアウトしました")
                    return []

                async def read_svg(container_id: str) -> Optional[TeamSvg]:
                    container = page.locator(f"#{container_id}")
                    if await container.count() == 0 or await container.locator("svg").first.count() == 0:
                        return None
                    try:
                        team_name = await container.evaluate(team_name_script) or "unknown"
                        svg_text = self._normalize_svg(await container.locator("svg").first.evaluate("el => el.outerHTML"))
                        return TeamSvg(team_name, svg_text)
                    except PlaywrightError:
                        logger.exception("PlaywrightでのSVG抽出に失敗しました: %s", container_id)
                        return None

                for container_id in ("chartHomeShots", "chartAwayShots"):
                    item = await read_svg(container_id)
                    if item:
                        found.append(item)

                if len(found) < 2:
                    charts = page.locator(ScraperSelectors.CHART_ANY_DIV)
                    for index in range(await charts.count()):
                        if len(found) >= 2:
                            break
                        container = charts.nth(index)
                        svg_locator = container.locator("svg").first
                        if await svg_locator.count() == 0:
                            continue
                        try:
                            team_name = await container.evaluate(team_name_script) or "unknown"
                            svg_text = self._normalize_svg(await svg_locator.evaluate("el => el.outerHTML"))
                            found.append(TeamSvg(team_name, svg_text))
                        except PlaywrightError:
                            logger.exception("Playwrightでの汎用SVG抽出に失敗しました: index=%s", index)
                return found[:2]
            finally:
                await browser.close()

    def download_background_image(self, timeout_sec: Optional[int] = None) -> Optional[Image.Image]:
        try:
            response = self.session.get(
                self.background_image_url,
                timeout=timeout_sec or self.network_config.timeout_sec,
            )
            response.raise_for_status()
            return Image.open(io.BytesIO(response.content)).convert("RGBA")
        except Exception:
            logger.exception("背景画像の取得に失敗しました")
            self._warn("シュートチャートの背景画像を取得できませんでした。ピッチ図なしで出力します")
            return None

    async def run_async(
        self,
        output_dir: Union[str, Path] = ".",
        timeout_sec: Optional[int] = None,
        html: Optional[str] = None,
    ) -> List[Path]:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        effective_timeout = timeout_sec or self.network_config.shot_timeout_sec
        if html is None:
            try:
                html = self.fetch_html(effective_timeout)
            except Exception:
                logger.exception("シュートチャート用HTML取得に失敗しました")
                self._warn("シュートチャート用のHTMLを取得できませんでした")
                return []

        svgs = self.extract_from_static_html(html)
        self.extraction_source = "静的HTML"
        if not svgs:
            logger.info("静的HTMLからSVGを取得できないため Playwright にフォールバックします")
            svgs = await self.extract_via_playwright(effective_timeout)
            self.extraction_source = "Playwright"
        if len(svgs) < 2:
            self._warn("シュートチャートのSVGを %s 個しか取得できませんでした（想定2個）", len(svgs))

        background = self.download_background_image(effective_timeout)
        saved_paths: List[Path] = []
        for index, item in enumerate(svgs):
            safe_team_name = re.sub(r'[\\/:*?"<>|\s]+', "_", item.team_name)
            path = output_path / f"shot_chart_{index}_{safe_team_name}.png"
            try:
                size_kwargs = (
                    {"parent_width": background.size[0], "parent_height": background.size[1]}
                    if background else {}
                )
                png_bytes = cairosvg.svg2png(bytestring=item.svg_text.encode("utf-8"), **size_kwargs)
                foreground = Image.open(io.BytesIO(png_bytes)).convert("RGBA")
                if background:
                    canvas = background.copy()
                    resized = foreground.resize(canvas.size, Image.Resampling.LANCZOS) if foreground.size != canvas.size else foreground
                    canvas.alpha_composite(resized)
                    canvas.save(path)
                else:
                    foreground.save(path)
                saved_paths.append(path)
            except Exception:
                logger.exception("シュートチャートPNG化に失敗しました: %s", path)
                self._warn("シュートチャートのPNG化に失敗しました: %s", item.team_name)
        return saved_paths

    @staticmethod
    def _extract_team_name(box) -> str:
        if not box:
            return "unknown_team"
        heading = box.find("h4", class_="tName")
        return heading.get_text(strip=True) if heading else "unknown_team"

    @staticmethod
    def _normalize_svg(svg_text: str) -> str:
        """xmlns の補完と、url(...#id) の参照をフラグメントだけに戻す2段階。"""
        with_namespace = svg_text
        if "xmlns=" not in svg_text:
            with_namespace = re.sub(
                r"<svg(\s)", r'<svg xmlns="http://www.w3.org/2000/svg"\1', svg_text, count=1, flags=re.IGNORECASE
            )
        return re.sub(r"url\((?:[^\)#]+)?#([^)]+)\)", r"url(#\1)", with_namespace, flags=re.IGNORECASE)

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [6/8] チャート生成
# ------------------------------------------------------------
# タイムライン（保持率＋シュート／ゴール）とレーダーチャート（CBP）。
# Ver.9 では CBPランキングの取得先カテゴリを J1/J2/J3 で切り替えられる
# ようにした。auto はランキング表に自チームが載っているカテゴリを
# 実測で判定する。
# ============================================================


@dataclass(frozen=True)
class TimelineBin:
    timebase: str
    home_poss: float
    away_poss: float
    home_shots: int
    away_shots: int
    home_goals: int
    away_goals: int


@dataclass(frozen=True)
class PreparedRadarChart:
    team1: str
    team2: str
    title: str
    team1_abbr: str
    team2_abbr: str
    labels: List[str]
    angles: List[float]
    team1_rel: List[float]
    team2_rel: List[float]
    league_rel: List[float]
    table_df: pd.DataFrame


@dataclass(frozen=True)
class CbpFetchResult:
    team1_avg: Dict[str, float]
    team2_avg: Dict[str, float]
    league_avg: Dict[str, float]
    team1_abbr: str
    team2_abbr: str
    team1_rel: Dict[str, float]
    team2_rel: Dict[str, float]


@dataclass(frozen=True)
class CbpSnapshot:
    category: CbpCategory
    league_avg: Dict[str, float]
    team_values: Dict[str, Dict[str, float]]


# ------------------------------------------------------------
# タイムライン
# ------------------------------------------------------------
class TimelineChartGenerator:
    TIMEBASE_PATTERN = re.compile(r"^\d{2}-\d{2}$")

    def __init__(
        self,
        html: str,
        club_config: ClubConfig,
        style_config: Optional[TimelineStyleConfig] = None,
        my_team: str = "",
        my_team_color: str = DEFAULT_MY_TEAM_COLOR,
        opponent_team_color: str = DEFAULT_OPPONENT_TEAM_COLOR,
        report: Optional[RunReport] = None,
    ):
        self.html = html
        self.club_config = club_config
        self.style_config = style_config or TimelineStyleConfig()
        self.my_team = my_team
        self.my_team_color = validate_hex_color(my_team_color, DEFAULT_MY_TEAM_COLOR)
        self.opponent_team_color = validate_hex_color(opponent_team_color, DEFAULT_OPPONENT_TEAM_COLOR)
        self.report = report

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    def generate_chart(self, out_path: str) -> bool:
        try:
            home_name, away_name, bins = self.parse_bins(self.html)
            if not bins:
                self._warn("タイムラインのbinが取得できませんでした")
                return False
            self.plot(home_name, away_name, bins, out_path)
            return True
        except Exception:
            logger.exception("タイムライン描画に失敗しました")
            return False

    @staticmethod
    def _merge_count(old_value: int, new_value: int) -> int:
        return max(int(old_value or 0), int(new_value or 0))

    @staticmethod
    def _parse_html(html: str) -> BeautifulSoup:
        try:
            return BeautifulSoup(html, "lxml")
        except Exception:
            logger.warning(
                "lxmlが使えないためhtml.parserで解析します。タイムラインのシュート数が0になる可能性があります"
            )
            return BeautifulSoup(html, "html.parser")

    @staticmethod
    def _direct_tds(row) -> List[Any]:
        return [child for child in row.children if getattr(child, "name", None) == "td"]

    @classmethod
    def _find_direct_td(cls, row, class_name: str):
        return next((td for td in cls._direct_tds(row) if class_name in (td.get("class") or [])), None)

    @classmethod
    def _apply_shot_counts(
        cls,
        bucket: Dict[str, Dict[str, Any]],
        label: Optional[str],
        home_cell,
        away_cell,
    ) -> None:
        if not label or home_cell is None or away_cell is None:
            return
        home_shots, home_goals = cls._count_shots_goals_static(home_cell)
        away_shots, away_goals = cls._count_shots_goals_static(away_cell)
        entry = bucket[label]
        entry["home_shots"] = cls._merge_count(entry["home_shots"], home_shots)
        entry["away_shots"] = cls._merge_count(entry["away_shots"], away_shots)
        entry["home_goals"] = cls._merge_count(entry["home_goals"], home_goals)
        entry["away_goals"] = cls._merge_count(entry["away_goals"], away_goals)

    @classmethod
    def aggregate_timeline_bins(cls, soup: BeautifulSoup) -> Tuple[str, str, List[TimelineBin]]:
        home_name, away_name = cls._extract_team_names_static(soup)
        bucket: Dict[str, Dict[str, Any]] = {}

        def ensure(label: str) -> Dict[str, Any]:
            if label not in bucket:
                bucket[label] = {
                    "timebase": label,
                    "home_poss": None,
                    "away_poss": None,
                    "home_shots": 0,
                    "away_shots": 0,
                    "home_goals": 0,
                    "away_goals": 0,
                }
            return bucket[label]

        for box in soup.select(ScraperSelectors.BOX_TIMELINE):
            table = box.find("table")
            if not table:
                continue
            current_label: Optional[str] = None
            current_home_shot_cell = None
            current_away_shot_cell = None
            counts_applied = False
            for row in table.find_all("tr"):
                time_cell = cls._find_direct_td(row, ScraperSelectors.TIMEBASE_CLASS)
                if time_cell is not None:
                    if current_label and not counts_applied:
                        cls._apply_shot_counts(bucket, current_label, current_home_shot_cell, current_away_shot_cell)
                    label = time_cell.get_text(strip=True)
                    current_label = label if cls.TIMEBASE_PATTERN.match(label) else None
                    current_home_shot_cell = None
                    current_away_shot_cell = None
                    counts_applied = False
                    if current_label:
                        ensure(current_label)
                if not current_label:
                    continue

                home_shot_cell = cls._find_direct_td(row, ScraperSelectors.HOME_SHOT_CLASS)
                away_shot_cell = cls._find_direct_td(row, ScraperSelectors.AWAY_SHOT_CLASS)
                if home_shot_cell is not None:
                    current_home_shot_cell = home_shot_cell
                if away_shot_cell is not None:
                    current_away_shot_cell = away_shot_cell

                home_poss_cell = cls._find_direct_td(row, ScraperSelectors.HOME_POSS_CLASS)
                away_poss_cell = cls._find_direct_td(row, ScraperSelectors.AWAY_POSS_CLASS)
                if home_poss_cell is not None and away_poss_cell is not None:
                    entry = ensure(current_label)
                    home_poss = cls._to_float(home_poss_cell)
                    away_poss = cls._to_float(away_poss_cell)
                    if entry["home_poss"] is None and home_poss is not None:
                        entry["home_poss"] = home_poss
                    if entry["away_poss"] is None and away_poss is not None:
                        entry["away_poss"] = away_poss

                title_cell = cls._find_direct_td(row, ScraperSelectors.DATA_TTL_CLASS)
                if title_cell is not None:
                    title_text = title_cell.get_text(strip=True).lower()
                    if any(keyword in title_text for keyword in ("shot", "シュート")):
                        cls._apply_shot_counts(bucket, current_label, current_home_shot_cell, current_away_shot_cell)
                        counts_applied = True
            if current_label and not counts_applied:
                cls._apply_shot_counts(bucket, current_label, current_home_shot_cell, current_away_shot_cell)

        bins = sorted(
            [
                TimelineBin(**entry)
                for entry in bucket.values()
                if entry["home_poss"] is not None and entry["away_poss"] is not None
            ],
            key=lambda item: int(item.timebase.split("-")[0]),
        )
        return home_name, away_name, bins

    def parse_bins(self, html: str) -> Tuple[str, str, List[TimelineBin]]:
        soup = self._parse_html(html)
        return self.aggregate_timeline_bins(soup)

    @staticmethod
    def _is_hidden(element) -> bool:
        style = (element.get("style") or "").lower().replace(" ", "")
        return any(keyword in style for keyword in ("display:none", "visibility:hidden"))

    @staticmethod
    def _extract_explicit_counts_from_attrs(cell) -> Optional[Tuple[int, int]]:
        for node in [cell] + cell.find_all(True):
            for attribute in ("title", "aria-label", "data-original-title", "data-title"):
                text = str(node.get(attribute) or "").strip()
                if not text:
                    continue
                match = re.search(r"シュート\s*[:：]?\s*(\d+).*?ゴール\s*[:：]?\s*(\d+)", text)
                if match:
                    shots = int(match.group(1))
                    return shots, min(int(match.group(2)), shots)
                match = re.search(r"shots?\s*[:=]?\s*(\d+).*?goals?\s*[:=]?\s*(\d+)", text, re.IGNORECASE)
                if match:
                    shots = int(match.group(1))
                    return shots, min(int(match.group(2)), shots)
        return None

    @classmethod
    def _collect_marker_imgs_from_direct_children(cls, cell) -> List[Any]:
        images: List[Any] = []
        nodes = [child for child in cell.children if getattr(child, "name", None) in {"span", "img", "i", "a", "em"}]
        for node in nodes:
            if cls._is_hidden(node):
                continue
            if node.name == "img":
                images.append(node)
            else:
                images.extend(image for image in node.find_all("img") if not cls._is_hidden(image))
        return images

    @classmethod
    def _count_shots_goals_static(cls, cell) -> Tuple[int, int]:
        if not cell:
            return 0, 0
        explicit = cls._extract_explicit_counts_from_attrs(cell)
        if explicit:
            return explicit
        direct_nodes = [child for child in cell.children if getattr(child, "name", None) in {"span", "img", "i", "a", "em"}]
        direct_spans = [node for node in direct_nodes if node.name == "span" and not cls._is_hidden(node)]
        images = cls._collect_marker_imgs_from_direct_children(cell)
        goals = 0
        for image in images:
            meta = f"{image.get('alt')} {image.get('title')} {image.get('aria-label')} {image.get('src')}".lower()
            if any(keyword in meta for keyword in ("goal", "ゴール")):
                goals += 1
        shots = len(direct_spans) + len(images)
        if shots == 0:
            text = cell.get_text(" ", strip=True)
            if match := re.search(r"(?:shots?|シュート)\s*[:：]?\s*(\d+)", text, re.IGNORECASE):
                shots = int(match.group(1))
            if match := re.search(r"(?:goals?|ゴール)\s*[:：]?\s*(\d+)", text, re.IGNORECASE):
                goals = int(match.group(1))
        return shots, min(goals, shots)

    @staticmethod
    def _hex_to_rgb01(hex_color: str) -> Tuple[float, float, float]:
        digits = hex_color.strip().lstrip("#")
        if len(digits) == 3:
            digits = "".join(char * 2 for char in digits)
        if len(digits) != 6:
            return (0.0, 0.0, 0.0)
        return (
            int(digits[0:2], 16) / 255.0,
            int(digits[2:4], 16) / 255.0,
            int(digits[4:6], 16) / 255.0,
        )

    @classmethod
    def _relative_luminance(cls, hex_color: str) -> float:
        red, green, blue = cls._hex_to_rgb01(hex_color)

        def linearize(value: float) -> float:
            return value / 12.92 if value <= 0.04045 else ((value + 0.055) / 1.055) ** 2.4

        return 0.2126 * linearize(red) + 0.7152 * linearize(green) + 0.0722 * linearize(blue)

    @classmethod
    def _halo_and_inner(cls, bar_hex: str, threshold: float = 0.42) -> Tuple[str, str]:
        return ("#ffffff", "#000000") if cls._relative_luminance(bar_hex) < threshold else ("#000000", "#ffffff")

    @staticmethod
    def _emoji_fontprop():
        try:
            from matplotlib import font_manager, ft2font
            target_codepoint = 0x26BD
            candidates: List[str] = [
                "/usr/share/fonts/truetype/noto/NotoSansSymbols2-Regular.ttf",
                "/usr/share/fonts/truetype/noto/NotoSansSymbols-Regular.ttf",
                "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            ]
            for extension in ("ttf", "otf"):
                try:
                    for font_path in font_manager.findSystemFonts(fontext=extension):
                        name = Path(font_path).name.lower()
                        if any(key in name for key in ("notosanssymbols2", "notosanssymbols", "dejavusans")):
                            candidates.append(font_path)
                except Exception:
                    logger.debug("フォント探索中に例外", exc_info=True)
            seen: set = set()
            for font_path in candidates:
                font_path = str(font_path)
                if font_path in seen:
                    continue
                seen.add(font_path)
                if not Path(font_path).exists():
                    continue
                try:
                    face = ft2font.FT2Font(font_path)
                    if target_codepoint in face.get_charmap():
                        return font_manager.FontProperties(fname=font_path)
                except Exception:
                    logger.debug("フォント読込失敗: %s", font_path, exc_info=True)
                    continue
        except Exception:
            logger.debug("絵文字フォント解決に失敗", exc_info=True)
        return None

    def resolve_timeline_bar_colors(self, home_name: str, away_name: str) -> Tuple[str, str]:
        home_abbr = self.club_config.resolve_team_abbr(home_name)
        away_abbr = self.club_config.resolve_team_abbr(away_name)
        my_abbr = self.club_config.resolve_team_abbr(self.my_team) if self.my_team else ""
        if my_abbr and home_abbr == my_abbr:
            return self.my_team_color, self.opponent_team_color
        if my_abbr and away_abbr == my_abbr:
            return self.opponent_team_color, self.my_team_color
        home_color = self.club_config.get_club_color(home_name, Kit.HOME, "#e60012")
        away_color = self.club_config.get_club_color(away_name, Kit.AWAY, "#f6b300")
        return home_color, away_color

    def plot(self, home_name: str, away_name: str, bins: List[TimelineBin], out_path: str) -> None:
        from matplotlib.lines import Line2D
        from matplotlib.legend_handler import HandlerBase
        from matplotlib.text import Text

        labels = [item.timebase for item in bins]
        home_possession = np.array([item.home_poss for item in bins])
        away_possession = np.array([item.away_poss for item in bins])
        home_color, away_color = self.resolve_timeline_bar_colors(home_name, away_name)
        home_halo, home_inner = self._halo_and_inner(home_color)
        away_halo, away_inner = self._halo_and_inner(away_color)
        emoji_font = self._emoji_fontprop()

        figure, axes = plt.subplots(figsize=self.style_config.timeline_figsize)
        axes.bar(
            range(len(labels)), home_possession, self.style_config.bar_width,
            label=home_name, color=home_color,
            edgecolor="black" if home_color.lower() == "#ffffff" else None,
        )
        axes.bar(
            range(len(labels)), away_possession, self.style_config.bar_width,
            bottom=home_possession, label=away_name, color=away_color,
            edgecolor="black" if away_color.lower() == "#ffffff" else None,
        )
        axes.set(
            ylim=(0, 100), ylabel="保持率(%)",
            xticks=range(len(labels)), xticklabels=labels,
            title=f"{home_name} vs {away_name} : タイムライン",
        )
        axes.axhline(50, color="black", linewidth=2)
        axes.grid(axis="y", linestyle="--", alpha=0.4)

        def draw_circles(x_index, y_values, edge_color, inner_color,
                         outer_size=90, outer_width=3.0, inner_size=55, inner_width=1.2):
            if y_values is None or len(y_values) == 0:
                return
            axes.scatter([x_index] * len(y_values), y_values, s=outer_size, facecolors="none",
                         edgecolors=edge_color, linewidths=outer_width, zorder=6)
            axes.scatter([x_index] * len(y_values), y_values, s=inner_size, facecolors="none",
                         edgecolors=inner_color, linewidths=inner_width, zorder=7)

        def draw_goals(x_index, y_values, pedestal: bool = False):
            if y_values is None or len(y_values) == 0:
                return
            pedestal_size = (self.style_config.goal_fontsize * self.style_config.goal_pedestal_scale) ** 2
            for y_value in y_values:
                if pedestal:
                    axes.scatter([x_index], [y_value], s=pedestal_size, marker="o", facecolors="#ffffff",
                                 edgecolors="#000000", linewidths=0.5, zorder=9)
                text_kwargs = dict(ha="center", va="center", fontsize=self.style_config.goal_fontsize,
                                   zorder=10, clip_on=True)
                if emoji_font is not None:
                    text_kwargs["fontproperties"] = emoji_font
                axes.text(x_index, y_value, "⚽", **text_kwargs)

        for index, item in enumerate(bins):
            home_positions = self._calc_stacked_pos(
                0.0, float(item.home_poss), item.home_shots,
                self.style_config.stack_step, self.style_config.stack_margin, False,
            )
            if item.home_shots > 0:
                home_goal_count = min(item.home_goals, item.home_shots)
                draw_goals(index, home_positions[:home_goal_count], pedestal=True)
                draw_circles(index, home_positions[home_goal_count:], home_halo, home_inner)
            away_positions = self._calc_stacked_pos(
                float(item.home_poss), 100.0, item.away_shots,
                self.style_config.stack_step, self.style_config.stack_margin, True,
            )
            if item.away_shots > 0:
                away_goal_count = min(item.away_goals, item.away_shots)
                draw_goals(index, away_positions[:away_goal_count], pedestal=True)
                draw_circles(index, away_positions[away_goal_count:], away_halo, away_inner)

        class _SoccerBallHandle:
            def __init__(self, fontprop=None, text="⚽"):
                self.fontprop = fontprop
                self.text = text

        class _SoccerBallHandler(HandlerBase):
            def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
                kwargs = dict(
                    x=xdescent + width / 2.0, y=ydescent + height / 2.0, text=orig_handle.text,
                    ha="center", va="center", fontsize=fontsize * 1.05, transform=trans,
                )
                if orig_handle.fontprop is not None:
                    kwargs["fontproperties"] = orig_handle.fontprop
                return [Text(**kwargs)]

        team_handles, team_labels = axes.get_legend_handles_labels()
        shot_handle = Line2D([0], [0], marker="o", linestyle="None", markerfacecolor="none",
                             markeredgecolor="black", markersize=9, label="シュート")
        goal_handle = _SoccerBallHandle(fontprop=emoji_font, text="⚽")
        axes.legend(
            handles=team_handles + [shot_handle, goal_handle],
            labels=team_labels + ["シュート", "ゴール"],
            handler_map={_SoccerBallHandle: _SoccerBallHandler()},
            loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=4,
        )
        figure.tight_layout()
        figure.savefig(out_path, dpi=self.style_config.timeline_dpi)
        plt.close(figure)

    @staticmethod
    def _extract_team_names_static(soup: BeautifulSoup) -> Tuple[str, str]:
        home_name, away_name = "HOME", "AWAY"
        compare_div = soup.select_one(ScraperSelectors.TBL_COMPARE)
        if not compare_div:
            return home_name, away_name
        em_tags = compare_div.find_all("em")
        if len(em_tags) >= 5:
            def pick(em_tag) -> str:
                span = em_tag.select_one(ScraperSelectors.TEAM_NAME_DSKTP)
                return span.get_text(strip=True) if span else em_tag.get_text(strip=True)

            home_name = pick(em_tags[1]) or home_name
            away_name = pick(em_tags[4]) or away_name
        return home_name, away_name

    @staticmethod
    def _to_float(cell) -> Optional[float]:
        match = re.search(r"(\d+(?:\.\d+)?)", cell.get_text(strip=True))
        return float(match.group(1)) if match else None

    @staticmethod
    def _calc_stacked_pos(
        y_start: float, y_end: float, count: int,
        step: float, margin: float, from_top: bool,
    ) -> np.ndarray:
        if count <= 0:
            return np.array([])
        usable_span = max(0.0, (y_end - y_start) - 2 * margin)
        if usable_span <= 0:
            return np.linspace(y_end if from_top else y_start, y_start if from_top else y_end, count)
        actual_step = min(step, usable_span / (count - 1) if count > 1 else step)
        origin = (y_end - margin) if from_top else (y_start + margin)
        if from_top:
            return origin - np.arange(count) * actual_step
        return origin + np.arange(count) * actual_step


# ------------------------------------------------------------
# レーダーチャート（CBP）
# ------------------------------------------------------------
class RadarChartGenerator:
    CBP_KEYS: Final[Tuple[str, ...]] = ("OFFENSE", "PASS", "CROSS", "DRIBBLE", "SHOT", "GAIN", "DEFENSE")
    CBP_KEYS_LABEL: Final[Dict[str, str]] = {
        "OFFENSE": "攻撃",
        "PASS": "パス",
        "CROSS": "クロス",
        "DRIBBLE": "ドリブル",
        "SHOT": "シュート",
        "GAIN": "奪取",
        "DEFENSE": "守備",
    }
    # Ver.8 では j1001 固定のURLを7本べた書きしていた。カテゴリを
    # 差し替えられるようテンプレート化する。
    CBP_RANKING_URL_TEMPLATE: Final[str] = "https://www.football-lab.jp/summary/cbp_ranking/{code}?year={year}&data={metric}"
    CBP_METRIC_PARAMS: Final[Dict[str, str]] = {
        "OFFENSE": "offense",
        "PASS": "pass",
        "CROSS": "cross",
        "DRIBBLE": "dribble",
        "SHOT": "shot",
        "GAIN": "gain",
        "DEFENSE": "defense",
    }

    def __init__(
        self,
        club_config: ClubConfig,
        session: requests.Session,
        network_config: Optional[NetworkConfig] = None,
        style_config: Optional[RadarStyleConfig] = None,
        cbp_config: Optional[CbpConfig] = None,
        my_team: str = "",
        my_team_color: str = DEFAULT_MY_TEAM_COLOR,
        opponent_team_color: str = DEFAULT_OPPONENT_TEAM_COLOR,
        report: Optional[RunReport] = None,
    ):
        self.club_config = club_config
        self.session = session
        self.network_config = network_config or NetworkConfig()
        self.style_config = style_config or RadarStyleConfig()
        self.cbp_config = cbp_config or CbpConfig()
        self.my_team = my_team
        self.my_team_color = validate_hex_color(my_team_color, DEFAULT_MY_TEAM_COLOR)
        self.opponent_team_color = validate_hex_color(opponent_team_color, DEFAULT_OPPONENT_TEAM_COLOR)
        self.report = report
        self._cbp_snapshot: Optional[CbpSnapshot] = None
        self._page_cache: Dict[Tuple[str, str], Tuple[float, int, Dict[str, float], Dict[str, Any]]] = {}

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    # ---- URLとカテゴリ ----------------------------------------
    def ranking_url(self, key: str, category: CbpCategory) -> str:
        return self.CBP_RANKING_URL_TEMPLATE.format(
            code=category.ranking_code,
            year=self.cbp_config.year,
            metric=self.CBP_METRIC_PARAMS[key],
        )

    def resolve_category(self) -> CbpCategory:
        """指定があればそれを使い、無ければランキング表に自チームが
        載っているカテゴリを実測で選ぶ。"""
        configured = self.cbp_config.category
        if configured is not None:
            if self.my_team and not self._team_in_category(self.my_team, configured):
                self._warn(
                    "指定カテゴリ %s のCBPランキングに %s が見つかりません。リーグ平均が実態と合っていない可能性があります",
                    configured.display_name, self.my_team,
                )
            return configured

        if not self.my_team:
            self._warn("自チーム名が空のためCBPカテゴリを判定できません。J1として扱います")
            return CbpCategory.J1

        for candidate in self.cbp_config.auto_detect_order:
            if self._team_in_category(self.my_team, candidate):
                logger.info("CBPカテゴリを自動判定しました: %s（%s）", candidate.display_name, self.my_team)
                return candidate

        self._warn(
            "%s をJ1/J2/J3のどのCBPランキングにも見つけられませんでした。J1として扱います",
            self.my_team,
        )
        return CbpCategory.J1

    def _team_in_category(self, team: str, category: CbpCategory) -> bool:
        try:
            _, _, team_values, _ = self._fetch_ranking_page("OFFENSE", category)
        except Exception:
            logger.exception("CBPカテゴリ判定の取得に失敗しました: category=%s", category.display_name)
            return False
        return self.club_config.resolve_team_abbr(team) in team_values

    # ---- 取得 ------------------------------------------------
    @staticmethod
    def _safe_float(value: Any, default: float = 0.0) -> float:
        try:
            return float(str(value).replace(",", "").strip())
        except (TypeError, ValueError):
            return default

    @classmethod
    def _extract_cbp_row_value(cls, cells) -> float:
        return cls._safe_float(cells[4].text if len(cells) >= 5 else 0.0)

    @staticmethod
    def _extract_scraped_team_name(cells) -> str:
        if len(cells) < 3:
            return ""
        team_span = cells[2].find("span", class_="dsktp")
        return team_span.text.strip() if team_span else ""

    def _fetch_ranking_page(
        self, key: str, category: CbpCategory,
    ) -> Tuple[float, int, Dict[str, float], Dict[str, Any]]:
        """1ページぶんの集計。カテゴリ判定とスナップショット取得で
        同じページを2回引かないようキャッシュする。"""
        cache_key = (category.value, key)
        if cache_key in self._page_cache:
            return self._page_cache[cache_key]

        cbp_total = 0.0
        valid_rows = 0
        team_values: Dict[str, float] = {}
        stats: Dict[str, Any] = {"skipped_shape": 0, "skipped_non_team": 0, "unknown_alias": []}

        response = self.session.get(self.ranking_url(key, category), timeout=self.network_config.cbp_timeout_sec)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")
        table = soup.find("table", id=ScraperSelectors.CBP_RANKING_TABLE_ID)
        if not table or not (tbody := table.find("tbody")):
            logger.warning(
                "CBPランキング表が見つかりませんでした: category=%s, key=%s",
                category.display_name, key,
            )
            result = (cbp_total, valid_rows, team_values, stats)
            self._page_cache[cache_key] = result
            return result

        for row in tbody.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) < 5:
                stats["skipped_shape"] += 1
                continue

            # チーム行かどうかは、チームページへのリンクの有無という構造で判定する。
            # 別名テーブルへの依存を切り離すことで、集計行や区切り行だけを除外できる。
            if not row.find("a", href=True):
                stats["skipped_non_team"] += 1
                continue

            row_value = self._extract_cbp_row_value(cells)
            # チーム行である限り、別名が引けなくてもリーグ平均の母数には含める。
            cbp_total += row_value
            valid_rows += 1

            scraped_team = self._extract_scraped_team_name(cells)
            if not self.club_config.is_known_team(scraped_team):
                stats["unknown_alias"].append(scraped_team or "(名称取得不可)")
                continue
            team_values[self.club_config.resolve_team_abbr(scraped_team)] = row_value

        result = (cbp_total, valid_rows, team_values, stats)
        self._page_cache[cache_key] = result
        return result

    def fetch_cbp_snapshot(self, *, force: bool = False) -> CbpSnapshot:
        if self._cbp_snapshot is not None and not force:
            return self._cbp_snapshot

        category = self.resolve_category()
        logger.info(
            "レーダーチャート用のリーグCBPデータを取得します: カテゴリ=%s / 7ページ",
            category.display_name,
        )
        totals = {key: 0.0 for key in self.CBP_KEYS}
        counts = {key: 0 for key in self.CBP_KEYS}
        team_values: Dict[str, Dict[str, float]] = {key: {} for key in self.CBP_KEYS}
        all_stats: Dict[str, Dict[str, Any]] = {}
        failed_keys: List[str] = []

        max_workers = min(self.network_config.max_workers, len(self.CBP_KEYS))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self._fetch_ranking_page, key, category): key for key in self.CBP_KEYS}
            for future in as_completed(futures):
                key = futures[future]
                try:
                    total_value, count_value, values, stats = future.result()
                    totals[key] = total_value
                    counts[key] = count_value
                    team_values[key] = values
                    all_stats[key] = stats
                except requests.RequestException:
                    logger.exception("CBPデータ取得に失敗しました: key=%s", key)
                    failed_keys.append(key)
                except Exception:
                    logger.exception("CBPデータ解析に失敗しました: key=%s", key)
                    failed_keys.append(key)

        if failed_keys:
            self._warn("CBPランキングを取得できなかった項目があります: %s", failed_keys)

        league_avg = {
            key: (totals[key] / counts[key]) if counts[key] > 0 else 0.0
            for key in self.CBP_KEYS
        }
        self._log_cbp_snapshot(category, counts, team_values, all_stats, league_avg)
        self._cbp_snapshot = CbpSnapshot(category=category, league_avg=league_avg, team_values=team_values)
        return self._cbp_snapshot

    def _log_cbp_snapshot(
        self,
        category: CbpCategory,
        counts: Dict[str, int],
        team_values: Dict[str, Dict[str, float]],
        all_stats: Dict[str, Dict[str, Any]],
        league_avg: Dict[str, float],
    ) -> None:
        """チーム行の判定がどう効いたかを追跡できるようにする。
        チーム行数（=リーグ平均の母数）と、別名を引けたチーム数が一致していれば正常。"""
        lines = [
            "",
            f"--- CBPランキング取得確認（{category.display_name}） ---",
            f"{pad_display('項目', 12)}{pad_display('チーム行', 10, 'right')}"
            f"{pad_display('別名解決', 10, 'right')}{pad_display('非チーム行', 12, 'right')}"
            f"{pad_display('列不足行', 10, 'right')}   リーグ平均",
        ]
        row_counts: set = set()
        unresolved_notes: List[str] = []
        for key in self.CBP_KEYS:
            stats = all_stats.get(key, {})
            team_rows = counts.get(key, 0)
            resolved = len(team_values.get(key, {}))
            row_counts.add(team_rows)
            lines.append(
                f"{pad_display(self.CBP_KEYS_LABEL.get(key, key), 12)}"
                f"{team_rows:>10}{resolved:>10}"
                f"{stats.get('skipped_non_team', 0):>12}{stats.get('skipped_shape', 0):>10}"
                f"   {league_avg.get(key, 0.0):.3f}"
            )
            if unknown := stats.get("unknown_alias"):
                lines.append(f"    別名テーブル未登録のため相対値に使えないチーム: {unknown}")
            if team_rows and resolved != team_rows:
                unresolved_notes.append(f"{self.CBP_KEYS_LABEL.get(key, key)}({resolved}/{team_rows})")
        lines.append("----------------------------")
        logger.info("\n".join(lines))

        if unresolved_notes:
            self._warn(
                "CBPランキングで別名を解決できないチーム行があります（リーグ平均は全行で計算）: %s",
                ", ".join(unresolved_notes),
            )
        if len(row_counts) > 1:
            self._warn("CBPランキングの項目ごとにチーム行数が揃っていません: %s", sorted(row_counts))
        if 0 in row_counts:
            self._warn("CBPランキングでチーム行が0件の項目があります。表のHTML構造が変わった可能性があります")

    def _fetch_cbp_values(
        self,
        team1: str,
        team2: str,
        custom_team1_vals: Optional[Dict[str, float]] = None,
        custom_team2_vals: Optional[Dict[str, float]] = None,
        snapshot: Optional[CbpSnapshot] = None,
    ) -> CbpFetchResult:
        snapshot = snapshot or self.fetch_cbp_snapshot()
        team1_abbr = self.club_config.resolve_team_abbr(team1)
        team2_abbr = self.club_config.resolve_team_abbr(team2)

        team1_avg: Dict[str, float] = {}
        team2_avg: Dict[str, float] = {}
        missing_team1: List[str] = []
        missing_team2: List[str] = []
        for key in self.CBP_KEYS:
            values = snapshot.team_values.get(key, {})
            if team1_abbr in values:
                team1_avg[key] = values[team1_abbr]
            else:
                team1_avg[key] = 0.0
                missing_team1.append(key)
            if team2_abbr in values:
                team2_avg[key] = values[team2_abbr]
            else:
                team2_avg[key] = 0.0
                missing_team2.append(key)

        if custom_team1_vals:
            for key, value in custom_team1_vals.items():
                if key in team1_avg:
                    team1_avg[key] = float(value)
            missing_team1 = [key for key in missing_team1 if key not in custom_team1_vals]

        if custom_team2_vals:
            for key, value in custom_team2_vals.items():
                if key in team2_avg:
                    team2_avg[key] = float(value)
            missing_team2 = [key for key in missing_team2 if key not in custom_team2_vals]

        if missing_team1 or missing_team2:
            self._warn(
                "CBPランキング(%s)からチーム値を取得できない項目があります（0として描画）: %s=%s, %s=%s",
                snapshot.category.display_name, team1, missing_team1, team2, missing_team2,
            )

        team1_rel: Dict[str, float] = {}
        team2_rel: Dict[str, float] = {}
        for key in self.CBP_KEYS:
            average = snapshot.league_avg.get(key, 0.0)
            team1_rel[key] = (team1_avg[key] / average) if average > 0 else 0.0
            team2_rel[key] = (team2_avg[key] / average) if average > 0 else 0.0

        return CbpFetchResult(
            team1_avg=team1_avg,
            team2_avg=team2_avg,
            league_avg=dict(snapshot.league_avg),
            team1_abbr=team1_abbr,
            team2_abbr=team2_abbr,
            team1_rel=team1_rel,
            team2_rel=team2_rel,
        )

    def prepare_chart_data(
        self,
        team1: str,
        team2: str,
        title: Optional[str] = None,
        custom_team1_vals: Optional[Dict[str, float]] = None,
        custom_team2_vals: Optional[Dict[str, float]] = None,
        snapshot: Optional[CbpSnapshot] = None,
    ) -> Tuple[PreparedRadarChart, pd.DataFrame]:
        result = self._fetch_cbp_values(
            team1, team2,
            custom_team1_vals=custom_team1_vals,
            custom_team2_vals=custom_team2_vals,
            snapshot=snapshot,
        )

        table_rows = [
            {
                "項目": self.CBP_KEYS_LABEL[key],
                f"{team1} (生値)": round(result.team1_avg[key], 2),
                f"{team1} (相対値)": round(result.team1_rel[key], 2),
                f"{team2} (生値)": round(result.team2_avg[key], 2),
                f"{team2} (相対値)": round(result.team2_rel[key], 2),
            }
            for key in self.CBP_KEYS
        ]
        table_df = pd.DataFrame(table_rows)

        labels = [self.CBP_KEYS_LABEL[key] for key in self.CBP_KEYS]
        angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
        angles += angles[:1]

        team1_rel = [result.team1_rel[key] for key in self.CBP_KEYS]
        team2_rel = [result.team2_rel[key] for key in self.CBP_KEYS]
        league_rel = [1.0 for _ in self.CBP_KEYS]

        prepared = PreparedRadarChart(
            team1=team1,
            team2=team2,
            title=title if title else f"{team1} vs {team2} : CBP比較",
            team1_abbr=result.team1_abbr,
            team2_abbr=result.team2_abbr,
            labels=labels,
            angles=angles,
            team1_rel=team1_rel + team1_rel[:1],
            team2_rel=team2_rel + team2_rel[:1],
            league_rel=league_rel + league_rel[:1],
            table_df=table_df,
        )
        return prepared, table_df

    @staticmethod
    def compute_outer_limit(prepared: PreparedRadarChart) -> float:
        max_value = max(
            max(prepared.team1_rel[:-1], default=1.0),
            max(prepared.team2_rel[:-1], default=1.0),
            1.0,
        )
        return float(max(1, int(np.ceil(max_value))))

    @classmethod
    def compute_common_outer_limit(cls, prepared_list: List[PreparedRadarChart]) -> float:
        valid_list = [item for item in prepared_list if item is not None]
        if not valid_list:
            return 1.0
        return max(cls.compute_outer_limit(item) for item in valid_list)

    @staticmethod
    def _build_radial_ticks(radial_max: float) -> List[float]:
        if radial_max <= 1.0:
            return [0.5, 1.0]
        ticks = np.arange(0.5, radial_max + 0.001, 0.5).tolist()
        return [round(value, 1) for value in ticks]

    def resolve_radar_colors(self, prepared: PreparedRadarChart) -> Tuple[str, str]:
        my_abbr = self.club_config.resolve_team_abbr(self.my_team) if self.my_team else ""
        if my_abbr and prepared.team1_abbr == my_abbr:
            return self.my_team_color, self.opponent_team_color
        if my_abbr and prepared.team2_abbr == my_abbr:
            return self.opponent_team_color, self.my_team_color

        team1_color = self.club_config.get_club_color(prepared.team1, Kit.HOME, "#ff0000")
        team2_color = self.club_config.get_club_color(prepared.team2, Kit.AWAY, "#0000ff")
        return (
            "#808080" if team1_color.lower() == "#ffffff" else team1_color,
            "#808080" if team2_color.lower() == "#ffffff" else team2_color,
        )

    def render_chart(
        self,
        prepared: PreparedRadarChart,
        out_path: str,
        radial_max: Optional[float] = None,
    ) -> bool:
        try:
            if radial_max is None:
                radial_max = self.compute_outer_limit(prepared)

            figure = plt.figure(figsize=self.style_config.radar_figsize)
            axes = figure.add_subplot(111, polar=True)

            # 画像内でレーダーチャート本体を中央に配置
            axes.set_position([
                self.style_config.axes_left,
                self.style_config.axes_bottom,
                self.style_config.axes_width,
                self.style_config.axes_height,
            ])

            axes.set_theta_offset(np.pi / 2)
            axes.set_theta_direction(-1)
            axes.set_thetagrids(np.degrees(prepared.angles[:-1]), prepared.labels)

            team1_color, team2_color = self.resolve_radar_colors(prepared)

            axes.plot(prepared.angles, prepared.team1_rel, color=team1_color,
                      linewidth=self.style_config.team_line_width, label=f"{prepared.team1}のCBP")
            axes.fill(prepared.angles, prepared.team1_rel, color=team1_color,
                      alpha=self.style_config.team_fill_alpha)

            axes.plot(prepared.angles, prepared.team2_rel, color=team2_color,
                      linewidth=self.style_config.team_line_width, label=f"{prepared.team2}のCBP")
            axes.fill(prepared.angles, prepared.team2_rel, color=team2_color,
                      alpha=self.style_config.team_fill_alpha)

            axes.plot(prepared.angles, prepared.league_rel, color="green",
                      linewidth=self.style_config.league_line_width,
                      linestyle=self.style_config.league_linestyle, label="リーグ平均")

            axes.set_ylim(0, radial_max)

            ticks = self._build_radial_ticks(radial_max)
            axes.set_yticks(ticks)
            axes.set_yticklabels(
                [str(int(value)) if float(value).is_integer() else str(value) for value in ticks]
            )
            axes.set_rlabel_position(self.style_config.rlabel_position)

            axes.set_title(prepared.title, y=self.style_config.title_y,
                           fontsize=self.style_config.title_fontsize)

            # 凡例は図の下中央
            axes.legend(
                loc=self.style_config.legend_loc,
                bbox_to_anchor=(self.style_config.legend_bbox_x, self.style_config.legend_bbox_y),
                ncol=self.style_config.legend_ncol,
                frameon=self.style_config.legend_frameon,
                borderaxespad=self.style_config.legend_borderaxespad,
            )

            figure.savefig(
                out_path,
                dpi=self.style_config.radar_dpi,
                bbox_inches=self.style_config.savefig_bbox_inches,
                pad_inches=self.style_config.savefig_pad_inches,
            )
            plt.close(figure)
            return True
        except Exception:
            logger.exception("レーダーチャート描画に失敗しました: %s", out_path)
            return False

    def generate_chart(
        self,
        team1: str,
        team2: str,
        out_path: str,
        title: Optional[str] = None,
        custom_team1_vals: Optional[Dict[str, float]] = None,
        custom_team2_vals: Optional[Dict[str, float]] = None,
        radial_max: Optional[float] = None,
        snapshot: Optional[CbpSnapshot] = None,
    ) -> Tuple[bool, Optional[pd.DataFrame]]:
        prepared, table_df = self.prepare_chart_data(
            team1, team2, title=title,
            custom_team1_vals=custom_team1_vals,
            custom_team2_vals=custom_team2_vals,
            snapshot=snapshot,
        )
        succeeded = self.render_chart(prepared, out_path, radial_max=radial_max)
        return succeeded, table_df

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [7/8] Google Docs / Drive 出力
# ------------------------------------------------------------
# Ver.9 の変更点:
#  - 一時画像を専用フォルダに入れる（Ver.8 はマイドライブ直下）
#  - Docs への貼り付けが終わったら公開権限(anyone/reader)を剥奪する
#  - 保持期間を過ぎた一時画像をゴミ箱へ退避する
# 画像は insertInlineImage の時点で Docs 側にコピーされるため、
# 後から公開権限を外しても本文の表示は維持される。
# ============================================================


@dataclass
class DriveUpload:
    file_id: str
    permission_id: str
    name: str


class GoogleExporter:
    def __init__(
        self,
        docs_style: Optional[DocsStyleConfig] = None,
        drive_config: Optional[DriveConfig] = None,
        club_config: Optional[ClubConfig] = None,
        my_team: str = "",
        report: Optional[RunReport] = None,
    ):
        if not _IN_COLAB:
            raise RuntimeError("Google Colab 環境専用です。")
        auth.authenticate_user()
        credentials, _ = default()
        self.docs_service = build("docs", "v1", credentials=credentials)
        self.drive_service = build("drive", "v3", credentials=credentials)
        self.docs_style = docs_style or DocsStyleConfig()
        self.drive_config = drive_config or DriveConfig()
        self.club_config = club_config
        self.my_team = my_team
        self.report = report
        self._doc_cursors: Dict[str, int] = {}
        self._uploads: List[DriveUpload] = []
        self._temp_folder_id: Optional[str] = None
        self._temp_folder_resolved: bool = False

    def _warn(self, message: str, *args: Any) -> None:
        emit_warning(self.report, message, *args)

    # ---- API 実行 --------------------------------------------
    def _is_retryable_google_error(self, exc: Exception) -> bool:
        status = getattr(getattr(exc, "resp", None), "status", None)
        if status in (429, 500, 502, 503, 504):
            return True
        text = str(exc)
        return any(
            key in text
            for key in ("rateLimitExceeded", "userRateLimitExceeded", "RESOURCE_EXHAUSTED", "Quota exceeded")
        )

    def _execute_google_request(self, request: Any, *, label: str) -> Any:
        wait_sec = self.docs_style.google_api_initial_wait_sec
        last_exception: Optional[Exception] = None
        for attempt in range(1, self.docs_style.google_api_max_retries + 1):
            try:
                result = request.execute()
                time.sleep(self.docs_style.google_api_request_interval_sec)
                return result
            except Exception as exc:
                last_exception = exc
                if not self._is_retryable_google_error(exc) or attempt >= self.docs_style.google_api_max_retries:
                    raise
                logger.warning(
                    "%s がレート制限/一時エラーのため再試行します (%s/%s, %.1f秒待機)",
                    label, attempt, self.docs_style.google_api_max_retries, wait_sec,
                )
                time.sleep(wait_sec)
                wait_sec = min(wait_sec * 2, self.docs_style.google_api_max_wait_sec)
        if last_exception is not None:
            raise last_exception

    def _docs_create(self, body: Dict[str, Any]) -> Dict[str, Any]:
        return self._execute_google_request(self.docs_service.documents().create(body=body), label="Docs作成")

    def _docs_get(self, document_id: str) -> Dict[str, Any]:
        return self._execute_google_request(
            self.docs_service.documents().get(documentId=document_id), label="Docs取得"
        )

    def _docs_batch_update(self, document_id: str, requests_body: List[Dict[str, Any]]) -> Dict[str, Any]:
        return self._execute_google_request(
            self.docs_service.documents().batchUpdate(documentId=document_id, body={"requests": requests_body}),
            label="Docs更新",
        )

    # ---- Drive 一時フォルダ ------------------------------------
    def ensure_temp_folder(self) -> Optional[str]:
        """一時画像の置き場を用意する。Ver.8 は parents 未指定で
        マイドライブ直下に散らばっていた。"""
        if self._temp_folder_resolved:
            return self._temp_folder_id
        self._temp_folder_resolved = True
        folder_name = self.drive_config.temp_folder_name
        escaped_name = folder_name.replace("\\", "\\\\").replace("'", "\\'")
        query = (
            "mimeType='application/vnd.google-apps.folder' and trashed=false "
            f"and name='{escaped_name}' and 'root' in parents"
        )
        try:
            found = self._execute_google_request(
                self.drive_service.files().list(q=query, fields="files(id,name)", pageSize=10),
                label="Driveフォルダ検索",
            )
            files = found.get("files", [])
            if files:
                self._temp_folder_id = files[0].get("id")
            else:
                created = self._execute_google_request(
                    self.drive_service.files().create(
                        body={"name": folder_name, "mimeType": "application/vnd.google-apps.folder"},
                        fields="id",
                    ),
                    label="Driveフォルダ作成",
                )
                self._temp_folder_id = created.get("id")
                logger.info("Drive に一時画像フォルダを作成しました: %s", folder_name)
        except Exception:
            logger.exception("Drive一時フォルダの準備に失敗しました")
            self._warn("Drive の一時フォルダを準備できませんでした。画像はマイドライブ直下に置かれます")
            self._temp_folder_id = None
        return self._temp_folder_id

    def _drive_create_file(self, path: Union[str, Path], parents: List[str]) -> Dict[str, Any]:
        return self._execute_google_request(
            self.drive_service.files().create(
                body={"name": Path(str(path)).name, "parents": parents},
                media_body=MediaFileUpload(str(path), mimetype="image/png"),
                fields="id, webContentLink",
            ),
            label="Driveファイル作成",
        )

    def _drive_create_permission(self, file_id: str) -> Dict[str, Any]:
        return self._execute_google_request(
            self.drive_service.permissions().create(
                fileId=file_id, body={"type": "anyone", "role": "reader"}, fields="id",
            ),
            label="Drive権限設定",
        )

    def upload_image_to_drive(self, path: Union[str, Path]) -> Optional[str]:
        try:
            folder_id = self.ensure_temp_folder()
            parents = [folder_id] if folder_id else []
            created = self._drive_create_file(path, parents)
            file_id = created.get("id")
            if not file_id:
                self._warn("Driveへのアップロード結果にファイルIDがありません: %s", path)
                return None
            permission = self._drive_create_permission(file_id)
            self._uploads.append(
                DriveUpload(file_id=file_id, permission_id=str(permission.get("id") or ""), name=Path(str(path)).name)
            )
            return created.get("webContentLink")
        except Exception:
            logger.exception("Driveアップロードに失敗しました: %s", path)
            self._warn("Driveへの画像アップロードに失敗しました: %s", Path(str(path)).name)
            return None

    def revoke_public_links(self) -> Tuple[int, int]:
        """Docs への貼り付け後に公開権限を外す。戻り値は (成功, 失敗)。"""
        succeeded = 0
        failed = 0
        for upload in self._uploads:
            if not upload.permission_id:
                failed += 1
                continue
            try:
                self._execute_google_request(
                    self.drive_service.permissions().delete(
                        fileId=upload.file_id, permissionId=upload.permission_id
                    ),
                    label="Drive公開権限の削除",
                )
                succeeded += 1
            except Exception:
                logger.exception("公開権限の削除に失敗しました: %s", upload.name)
                failed += 1
        if failed:
            self._warn("公開リンクを解除できなかった画像が %s 件あります。Drive で手動確認してください", failed)
        self._uploads.clear()
        return succeeded, failed

    def purge_old_temp_files(self, retention_days: Optional[int] = None) -> int:
        """保持期間を過ぎた一時画像をゴミ箱へ移す。完全削除はしない。"""
        days = self.drive_config.retention_days if retention_days is None else retention_days
        if days <= 0:
            return 0
        folder_id = self.ensure_temp_folder()
        if not folder_id:
            return 0
        threshold = (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%dT%H:%M:%SZ")
        query = (
            f"'{folder_id}' in parents and trashed=false "
            f"and mimeType != 'application/vnd.google-apps.folder' "
            f"and createdTime < '{threshold}'"
        )
        purged = 0
        try:
            listed = self._execute_google_request(
                self.drive_service.files().list(q=query, fields="files(id,name)", pageSize=200),
                label="Drive一時画像の検索",
            )
            for item in listed.get("files", []):
                file_id = item.get("id")
                if not file_id:
                    continue
                self._execute_google_request(
                    self.drive_service.files().update(fileId=file_id, body={"trashed": True}),
                    label="Drive一時画像のゴミ箱移動",
                )
                purged += 1
        except Exception:
            logger.exception("古い一時画像の整理に失敗しました")
            self._warn("Drive の古い一時画像を整理できませんでした")
        return purged

    # ---- Docs 本文 -------------------------------------------
    @staticmethod
    def _docs_text_len(text: str) -> int:
        # Google Docs API の index は UTF-16 code unit ベース
        return len(str(text).encode("utf-16-le")) // 2

    @staticmethod
    def _named_style_for(item_type: str) -> str:
        return {
            "title": "TITLE",
            "header": "HEADING_2",
            "sub_header": "HEADING_3",
        }.get(item_type, "NORMAL_TEXT")

    def create_doc(
        self,
        title: str,
        doc_structure: List[Dict[str, Any]],
        timeline_url: Optional[str] = None,
        shot_chart_data: Optional[List[Tuple[str, str]]] = None,
        radar_charts_data: Optional[List[Tuple[str, str]]] = None,
    ) -> str:
        document_id = self._docs_create({"title": title}).get("documentId")
        if not document_id:
            raise RuntimeError("Google Docs の documentId を取得できませんでした")
        self._doc_cursors[document_id] = 1
        document_url = f"https://docs.google.com/document/d/{document_id}/edit"
        logger.info("Docs: %s", document_url)

        for item in doc_structure:
            if item.get("type") == "table":
                self._insert_table_to_doc(document_id, item.get("content"))
            else:
                style = self._named_style_for(str(item.get("type")))
                self._append_text_block(document_id, str(item.get("content")), style)

        if radar_charts_data:
            for image_url, caption in radar_charts_data:
                if caption.strip():
                    self._append_text_block(document_id, caption, "HEADING_2")
                self._append_image_block(document_id, image_url, self.docs_style.radar_image_size_pt)

        if timeline_url:
            self._append_text_block(document_id, "タイムライン", "HEADING_2")
            self._append_image_block(document_id, timeline_url, self.docs_style.timeline_image_size_pt)

        if shot_chart_data:
            for image_url, caption in shot_chart_data:
                if caption.strip():
                    self._append_text_block(document_id, caption, "HEADING_2")
                self._append_image_block(document_id, image_url, self.docs_style.shot_image_size_pt)

        return document_url

    def _sync_document_cursor(self, document_id: str, document: Optional[Dict[str, Any]] = None) -> int:
        document = document or self._docs_get(document_id)
        body = document.get("body", {}).get("content", [])
        cursor = max(1, int(body[-1].get("endIndex", 2)) - 1) if body else 1
        self._doc_cursors[document_id] = cursor
        return cursor

    def _append_text_block(self, document_id: str, text: str, named_style: str) -> None:
        content = str(text or "")
        if not content.strip():
            return
        index = self._doc_cursors.get(document_id, 1)
        paragraph_text = f"{content}\n"
        text_length = self._docs_text_len(paragraph_text)
        self._docs_batch_update(document_id, [
            {"insertText": {"location": {"index": index}, "text": paragraph_text}},
            {"updateParagraphStyle": {
                "range": {"startIndex": index, "endIndex": index + text_length},
                "paragraphStyle": {"namedStyleType": named_style},
                "fields": "namedStyleType",
            }},
        ])
        self._doc_cursors[document_id] = index + text_length

    def _append_image_block(self, document_id: str, image_url: str, size_pt: Tuple[int, int]) -> None:
        index = self._doc_cursors.get(document_id, 1)
        self._docs_batch_update(document_id, [
            {"insertInlineImage": {
                "location": {"index": index},
                "uri": image_url,
                "objectSize": {
                    "height": {"magnitude": size_pt[1], "unit": "PT"},
                    "width": {"magnitude": size_pt[0], "unit": "PT"},
                },
            }},
            {"insertText": {"location": {"index": index + 1}, "text": "\n"}},
        ])
        self._doc_cursors[document_id] = index + 2

    # ---- Docs 表 ---------------------------------------------
    @staticmethod
    def _extract_bracket_team_name(value: Any) -> str:
        match = re.match(r"^\[(.+?)\]", str(value))
        return match.group(1) if match else ""

    @staticmethod
    def _strip_bracket_prefix(value: Any) -> str:
        return re.sub(r"^\[[^\]]+\]\s*", "", str(value)).replace(".1", "")

    def _team_cell_kind(self, value: Any) -> Optional[str]:
        """セルの文字列がチーム名なら "my" / "opponent" を返す。
        レーダー表の「名古屋グランパス (生値)」のような末尾の括弧は落として判定する。"""
        if not self.club_config:
            return None
        text = re.sub(r"[（(].*?[）)]\s*$", "", str(value or "")).strip()
        if not text or not self.club_config.is_known_team(text):
            return None
        if self.my_team and self.club_config.resolve_team_abbr(text) == self.club_config.resolve_team_abbr(self.my_team):
            return "my"
        return "opponent"

    def _detect_team_cells(self, df_out: pd.DataFrame) -> Dict[Tuple[int, int], str]:
        # チーム名が入るのは見出し行だけなので、選手名などの誤検知を避けるため先頭行に限定する。
        team_cells: Dict[Tuple[int, int], str] = {}
        if df_out.empty:
            return team_cells
        for column_index in range(df_out.shape[1]):
            if kind := self._team_cell_kind(df_out.iloc[0, column_index]):
                team_cells[(0, column_index)] = kind
        return team_cells

    def _prepare_table_for_docs(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[Tuple[int, int], str]]:
        df_out = self._build_docs_table_frame(df)
        return df_out, self._detect_team_cells(df_out)

    def _build_docs_table_frame(self, df: pd.DataFrame) -> pd.DataFrame:
        columns = [str(column) for column in df.columns]
        if len(columns) == 5 and columns[2] == "項目":
            left_team = self._extract_bracket_team_name(columns[0]) or self._extract_bracket_team_name(columns[1])
            right_team = self._extract_bracket_team_name(columns[3]) or self._extract_bracket_team_name(columns[4])
            header_rows = pd.DataFrame([
                ["", left_team, "", right_team, ""],
                [
                    self._strip_bracket_prefix(columns[0]),
                    self._strip_bracket_prefix(columns[1]),
                    columns[2],
                    self._strip_bracket_prefix(columns[3]),
                    self._strip_bracket_prefix(columns[4]),
                ],
            ], columns=columns)
            return pd.concat([header_rows, df.reset_index(drop=True)], ignore_index=True)
        if len(columns) == 7 and columns[3] == "項目":
            left_team = (
                self._extract_bracket_team_name(columns[0])
                or self._extract_bracket_team_name(columns[1])
                or self._extract_bracket_team_name(columns[2])
            )
            right_team = (
                self._extract_bracket_team_name(columns[4])
                or self._extract_bracket_team_name(columns[5])
                or self._extract_bracket_team_name(columns[6])
            )
            header_rows = pd.DataFrame([
                ["", "", left_team, "", right_team, "", ""],
                [
                    self._strip_bracket_prefix(columns[0]),
                    self._strip_bracket_prefix(columns[1]),
                    self._strip_bracket_prefix(columns[2]),
                    columns[3],
                    self._strip_bracket_prefix(columns[4]),
                    self._strip_bracket_prefix(columns[5]),
                    self._strip_bracket_prefix(columns[6]),
                ],
            ], columns=columns)
            return pd.concat([header_rows, df.reset_index(drop=True)], ignore_index=True)
        return pd.concat(
            [pd.DataFrame([df.columns], columns=df.columns), df.reset_index(drop=True)],
            ignore_index=True,
        )

    def _insert_table_to_doc(self, document_id: str, df: Optional[pd.DataFrame]) -> None:
        if df is None or not isinstance(df, pd.DataFrame) or df.empty:
            self._warn("空の表が渡されたためスキップしました")
            return
        df_out, team_cells = self._prepare_table_for_docs(df)
        row_count, column_count = df_out.shape
        insert_index = self._doc_cursors.get(document_id, 1)
        self._docs_batch_update(
            document_id,
            [{"insertTable": {"rows": int(row_count), "columns": int(column_count),
                              "location": {"index": insert_index}}}],
        )

        # 表のセル index は Docs 側で決まるため、ここだけ1回 documents.get() する。
        document = self._docs_get(document_id)
        body = document.get("body", {}).get("content", [])
        base_cursor = max(1, int(body[-1].get("endIndex", 2)) - 1) if body else insert_index
        tables = [element["table"] for element in body if "table" in element]
        table = tables[-1] if tables else None
        if not table:
            self._warn("挿入した表をDocs上で見つけられませんでした。セルの中身が空のままになります")
            self._doc_cursors[document_id] = base_cursor
            return

        cells = [
            (cell_data["content"][0].get("startIndex"), str(df_out.iloc[row_index, column_index]))
            for row_index, row_data in enumerate(table.get("tableRows", [])) if row_index < row_count
            for column_index, cell_data in enumerate(row_data.get("tableCells", []))
            if column_index < column_count
            and str(df_out.iloc[row_index, column_index]) not in ("", "nan")
            and cell_data["content"][0].get("startIndex")
        ]
        insert_requests: List[Dict[str, Any]] = [
            {"insertText": {"location": {"index": index}, "text": text}}
            for index, text in sorted(cells, key=lambda pair: pair[0], reverse=True)
        ]
        # 表の後ろの改行も同じ batchUpdate にまとめる。
        insert_requests.append({"insertText": {"endOfSegmentLocation": {}, "text": "\n"}})
        self._docs_batch_update(document_id, insert_requests)
        added_units = sum(self._docs_text_len(text) for _, text in cells) + 1
        self._doc_cursors[document_id] = base_cursor + added_units

        try:
            self._style_table(document_id, df_out, team_cells)
        except Exception:
            logger.exception("表のスタイル適用に失敗しました（内容は挿入済み）")
            self._warn("表の書式設定に失敗しました（内容は挿入済み）")

    @staticmethod
    def _hex_to_rgb_color(hex_color: str) -> Dict[str, Any]:
        digits = normalize_hex_color(hex_color).lstrip("#")
        if len(digits) == 3:
            digits = "".join(char * 2 for char in digits)
        if len(digits) != 6:
            digits = "000000"
        return {
            "red": int(digits[0:2], 16) / 255.0,
            "green": int(digits[2:4], 16) / 255.0,
            "blue": int(digits[4:6], 16) / 255.0,
        }

    def _estimate_text_width_pt(self, text: Any) -> float:
        """セル内改行が起きない最小幅の目安。全角は約1em、半角は約0.55emで見積もる。"""
        font_pt = self.docs_style.table_font_pt
        width = 0.0
        for char in str(text):
            width += font_pt if unicodedata.east_asian_width(char) in ("W", "F", "A") else font_pt * 0.55
        return width

    def _estimate_column_widths(self, df_out: pd.DataFrame) -> List[float]:
        style = self.docs_style
        widths: List[float] = []
        for column_index in range(df_out.shape[1]):
            longest = max(
                (self._estimate_text_width_pt(df_out.iloc[row_index, column_index])
                 for row_index in range(df_out.shape[0])),
                default=0.0,
            )
            widths.append(max(style.table_min_column_width_pt, longest + style.table_cell_padding_pt))
        total = sum(widths)
        # ページ幅を超える場合だけ縮める。ここでは改行が発生しうる。
        if total > style.table_max_total_width_pt and total > 0:
            scale = style.table_max_total_width_pt / total
            widths = [max(style.table_min_column_width_pt, width * scale) for width in widths]
        return widths

    def _style_table(self, document_id: str, df_out: pd.DataFrame, team_cells: Dict[Tuple[int, int], str]) -> None:
        # セルの index は文字挿入後にずれるため、ここで取り直す。
        document = self._docs_get(document_id)
        body = document.get("body", {}).get("content", [])
        table_elements = [element for element in body if "table" in element]
        if not table_elements:
            return
        element = table_elements[-1]
        table_start = element.get("startIndex")
        table_end = element.get("endIndex")
        table = element["table"]
        if table_start is None or table_end is None:
            return

        start_location = {"index": table_start}
        style_requests: List[Dict[str, Any]] = []

        if self.docs_style.table_align_center:
            style_requests.append({"updateParagraphStyle": {
                "range": {"startIndex": table_start, "endIndex": table_end},
                "paragraphStyle": {"alignment": "CENTER"},
                "fields": "alignment",
            }})

        for column_index, width in enumerate(self._estimate_column_widths(df_out)):
            style_requests.append({"updateTableColumnProperties": {
                "tableStartLocation": start_location,
                "columnIndices": [column_index],
                "tableColumnProperties": {
                    "widthType": "FIXED_WIDTH",
                    "width": {"magnitude": round(width, 1), "unit": "PT"},
                },
                "fields": "widthType,width",
            }})

        rows = table.get("tableRows", [])
        for (row_index, column_index), kind in team_cells.items():
            if row_index >= len(rows) or column_index >= len(rows[row_index].get("tableCells", [])):
                continue
            background = self.docs_style.my_team_cell_bg if kind == "my" else self.docs_style.opponent_team_cell_bg
            style_requests.append({"updateTableCellStyle": {
                "tableRange": {
                    "tableCellLocation": {
                        "tableStartLocation": start_location,
                        "rowIndex": row_index,
                        "columnIndex": column_index,
                    },
                    "rowSpan": 1,
                    "columnSpan": 1,
                },
                "tableCellStyle": {"backgroundColor": {"color": {"rgbColor": self._hex_to_rgb_color(background)}}},
                "fields": "backgroundColor",
            }})
            if kind != "my":
                continue
            # 濃い背景色なので文字色を明るくする
            content = rows[row_index]["tableCells"][column_index].get("content", [])
            if not content:
                continue
            cell_start = content[0].get("startIndex")
            cell_end = content[-1].get("endIndex")
            if cell_start is None or cell_end is None or cell_end - 1 <= cell_start:
                continue
            style_requests.append({"updateTextStyle": {
                "range": {"startIndex": cell_start, "endIndex": cell_end - 1},
                "textStyle": {
                    "foregroundColor": {
                        "color": {"rgbColor": self._hex_to_rgb_color(self.docs_style.my_team_cell_text)}
                    },
                    "bold": True,
                },
                "fields": "foregroundColor,bold",
            }})

        if style_requests:
            self._docs_batch_update(document_id, style_requests)

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# Konwenga Helper Ver.9.0.0  [8/8] オーケストレーションと実行
# ------------------------------------------------------------
# 成果物ごとの成否は RunReport に集約し、最後に必ず一覧を出す。
# 途中で何かが欠けても「完了」とだけ出る状態を無くすのが目的。
# ============================================================


def extract_current_match_cbp_values(
    doc_structure: List[Dict[str, Any]],
    label_to_key: Dict[str, str],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    home_values: Dict[str, float] = {}
    away_values: Dict[str, float] = {}
    for item in doc_structure:
        if item.get("type") != "table" or not isinstance(item.get("content"), pd.DataFrame):
            continue
        frame = item["content"]
        if "項目" not in frame.columns:
            continue
        matched_labels = sum(1 for label in label_to_key if label in frame["項目"].values)
        if matched_labels < 5:
            continue
        for _, row in frame.iterrows():
            label = row["項目"]
            if label in label_to_key:
                key = label_to_key[label]
                home_values[key] = RadarChartGenerator._safe_float(row.iloc[1], 0.0)
                away_values[key] = RadarChartGenerator._safe_float(row.iloc[3], 0.0)
        break
    return home_values, away_values


def build_doc_title(file_date: str) -> str:
    return f"report_{file_date}"


def build_radar_tables_block(title: str, frame: Optional[pd.DataFrame]) -> List[Dict[str, Any]]:
    if frame is None:
        return []
    return [{"type": "header", "content": title}, {"type": "table", "content": frame}]


def generate_timeline_artifact(
    scraper: FootballLabScraper,
    club_config: ClubConfig,
    exporter: GoogleExporter,
    app_config: AppConfig,
    report: RunReport,
) -> Tuple[Optional[str], str, str, List[TimelineBin]]:
    timeline_generator = TimelineChartGenerator(
        scraper.html_text,
        club_config,
        style_config=app_config.timeline_style,
        my_team=app_config.my_team,
        my_team_color=app_config.my_team_color,
        opponent_team_color=app_config.opponent_team_color,
        report=report,
    )
    # v7と同じく、タイムラインは html_text を lxml で解析し直した soup を使う。
    # scraper.soup は html.parser 製で、タイムライン表の td 入れ子が補正されず
    # td.homeShot / td.awayShot を tr の直下として拾えなくなる（シュート数が全て0になる）。
    home_name, away_name, bins = timeline_generator.parse_bins(scraper.html_text)

    if bins:
        detail_lines = ["", "--- タイムライン集計確認 ---"]
        detail_lines.extend(
            f"{item.timebase} {home_name}: shots={item.home_shots}, goals={item.home_goals}"
            f" / {away_name}: shots={item.away_shots}, goals={item.away_goals}"
            for item in bins
        )
        detail_lines.append("----------------------------")
        logger.info("\n".join(detail_lines))
    else:
        report.ng("タイムライン", "binを取得できませんでした")
        return None, home_name, away_name, bins

    timeline_file = f"timeline_{scraper.file_date}.png"
    timeline_url: Optional[str] = None
    try:
        timeline_generator.plot(home_name, away_name, bins, timeline_file)
        timeline_url = exporter.upload_image_to_drive(timeline_file)
    except Exception:
        logger.exception("タイムライン描画/アップロードに失敗しました")

    total_shots = sum(item.home_shots + item.away_shots for item in bins)
    if timeline_url:
        report.ok("タイムライン", f"{len(bins)}bin / シュート計{total_shots}本")
    else:
        report.ng("タイムライン", "画像の生成またはアップロードに失敗しました")
    return timeline_url, home_name, away_name, bins


def prepare_radar_artifacts(
    doc_structure: List[Dict[str, Any]],
    home_name: str,
    away_name: str,
    radar_generator: RadarChartGenerator,
    exporter: GoogleExporter,
    scraper: FootballLabScraper,
    app_config: AppConfig,
    report: RunReport,
) -> Tuple[List[Tuple[str, str]], List[Dict[str, Any]]]:
    label_to_key = {label: key for key, label in radar_generator.CBP_KEYS_LABEL.items()}
    home_cbp_values, away_cbp_values = extract_current_match_cbp_values(doc_structure, label_to_key)
    if not home_cbp_values:
        report.warn("試合レポートから今節のCBP実測値を拾えませんでした。今節レーダーはランキング値で描画します")

    radar_charts_data: List[Tuple[str, str]] = []
    radar_tables_data: List[Dict[str, Any]] = []

    try:
        snapshot = radar_generator.fetch_cbp_snapshot()
    except Exception:
        logger.exception("CBPスナップショットの取得に失敗しました")
        report.ng("CBPランキング", "取得に失敗しました")
        report.skip("レーダー(今節)", "CBPランキングが無いため")
        report.skip("レーダー(次節)", "CBPランキングが無いため")
        return radar_charts_data, radar_tables_data

    resolved_pages = sum(1 for key in radar_generator.CBP_KEYS if snapshot.team_values.get(key))
    report.ok(
        "CBPランキング",
        f"カテゴリ={snapshot.category.display_name} / {resolved_pages}/{len(radar_generator.CBP_KEYS)}項目",
    )

    current_title = f"今節: {home_name} vs {away_name} CBP比較"
    next_title = f"次節: {app_config.my_team} vs {app_config.next_team} CBP比較"

    prepared_current: Optional[PreparedRadarChart] = None
    prepared_next: Optional[PreparedRadarChart] = None
    frame_current: Optional[pd.DataFrame] = None
    frame_next: Optional[pd.DataFrame] = None

    try:
        prepared_current, frame_current = radar_generator.prepare_chart_data(
            home_name, away_name, title=current_title,
            custom_team1_vals=home_cbp_values, custom_team2_vals=away_cbp_values,
            snapshot=snapshot,
        )
    except Exception:
        logger.exception("今節レーダーチャート準備に失敗しました")

    if app_config.my_team and app_config.next_team:
        try:
            prepared_next, frame_next = radar_generator.prepare_chart_data(
                app_config.my_team, app_config.next_team, title=next_title, snapshot=snapshot,
            )
        except Exception:
            logger.exception("次節レーダーチャート準備に失敗しました")
    else:
        report.skip("レーダー(次節)", "自チームまたは次節対戦相手が未設定")

    prepared_list = [item for item in (prepared_current, prepared_next) if item is not None]
    common_outer_limit = RadarChartGenerator.compute_common_outer_limit(prepared_list) if prepared_list else 1.0
    logger.info("レーダーチャート共通最外周: %s", common_outer_limit)

    def emit_radar(
        prepared: Optional[PreparedRadarChart],
        frame: Optional[pd.DataFrame],
        file_name: str,
        caption: str,
        table_title: str,
        step_name: str,
    ) -> None:
        if prepared is None:
            report.ng(step_name, "チャートデータを準備できませんでした")
            return
        if not radar_generator.render_chart(prepared, file_name, radial_max=common_outer_limit):
            report.ng(step_name, "描画に失敗しました")
            return
        image_url = exporter.upload_image_to_drive(file_name)
        if image_url:
            radar_charts_data.append((image_url, caption))
            report.ok(step_name, caption)
        else:
            report.ng(step_name, "Driveへのアップロードに失敗しました")
        radar_tables_data.extend(build_radar_tables_block(table_title, frame))

    emit_radar(
        prepared_current, frame_current,
        f"radar_chart_current_{scraper.file_date}.png",
        current_title,
        f"【今節】{home_name} vs {away_name} CBPデータ",
        "レーダー(今節)",
    )
    if app_config.my_team and app_config.next_team:
        emit_radar(
            prepared_next, frame_next,
            f"radar_chart_next_{scraper.file_date}.png",
            next_title,
            f"【次節】{app_config.my_team} vs {app_config.next_team} CBPデータ",
            "レーダー(次節)",
        )
    return radar_charts_data, radar_tables_data


def generate_shot_chart_artifacts(
    scraper: FootballLabScraper,
    session: requests.Session,
    exporter: GoogleExporter,
    app_config: AppConfig,
    report: RunReport,
) -> List[Tuple[str, str]]:
    logger.info("シュートチャートを生成します")
    shot_chart_data: List[Tuple[str, str]] = []
    svg_exporter = FootballLabShotsSvgExporter(
        app_config.target_match_url,
        session=session,
        network_config=app_config.network,
        report=report,
    )
    try:
        shot_paths = run_in_notebook(
            svg_exporter.run_async(
                timeout_sec=app_config.network.shot_timeout_sec,
                html=scraper.html_text,
            )
        )
    except Exception:
        logger.exception("シュートチャート生成に失敗しました")
        report.ng("シュートチャート", "生成に失敗しました")
        return shot_chart_data

    for path in shot_paths:
        image_url = exporter.upload_image_to_drive(path)
        if not image_url:
            continue
        caption = f"{Path(path).stem.split('_', 3)[-1].replace('_', ' ')}のシュート"
        shot_chart_data.append((image_url, caption))

    if shot_chart_data:
        report.ok("シュートチャート", f"{len(shot_chart_data)}枚 / 取得元={svg_exporter.extraction_source}")
    else:
        report.ng("シュートチャート", "1枚も生成できませんでした")
    return shot_chart_data


def main() -> RunReport:
    report = RunReport()
    app_config = build_app_config(
        target_match_url=TARGET_MATCH_URL,
        my_team=MY_TEAM,
        next_team=NEXT_TEAM,
        my_team_color=MY_TEAM_COLOR,
        opponent_team_color=OPPONENT_TEAM_COLOR,
        cbp_category=CBP_RANKING_CATEGORY,
        cbp_year=CBP_RANKING_YEAR,
        drive_temp_folder_name=DRIVE_TEMP_FOLDER_NAME,
        revoke_public_links_after_export=REVOKE_PUBLIC_LINKS_AFTER_EXPORT,
        drive_retention_days=DRIVE_TEMP_RETENTION_DAYS,
    )
    logger.info("Football-Lab記事生成 Ver.%s", VERSION)
    logger.info("解析: %s", app_config.target_match_url)
    logger.info(
        "自チーム色: %s / 相手色: %s / CBPカテゴリ: %s",
        app_config.my_team_color,
        app_config.opponent_team_color,
        app_config.cbp.category.display_name if app_config.cbp.category else "auto",
    )

    session = build_session(app_config.network)
    scraper = FootballLabScraper(
        app_config.target_match_url,
        session=session,
        network_config=app_config.network,
        report=report,
    )
    if not scraper.fetch():
        report.ng("試合レポート取得", app_config.target_match_url)
        report.emit()
        return report
    report.ok("試合レポート取得", scraper.page_title)

    club_config = load_club_config()
    club_config, alias_report = augment_club_config_from_html(club_config, scraper.soup)
    log_club_config(club_config, alias_report, report)

    try:
        exporter = GoogleExporter(
            docs_style=app_config.docs_style,
            drive_config=app_config.drive,
            club_config=club_config,
            my_team=app_config.my_team,
            report=report,
        )
    except Exception:
        logger.exception("Google API の初期化に失敗しました")
        report.ng("Google認証", "Docs/Driveに接続できませんでした")
        report.emit()
        return report

    doc_structure = scraper.get_structure()
    table_count = sum(1 for item in doc_structure if item.get("type") == "table")
    if table_count:
        report.ok("試合レポート解析", f"表 {table_count}個")
    else:
        report.ng("試合レポート解析", "表を抽出できませんでした")

    timeline_url, home_name, away_name, _bins = generate_timeline_artifact(
        scraper, club_config, exporter, app_config, report
    )

    radar_generator = RadarChartGenerator(
        club_config,
        session=session,
        network_config=app_config.network,
        style_config=app_config.radar_style,
        cbp_config=app_config.cbp,
        my_team=app_config.my_team,
        my_team_color=app_config.my_team_color,
        opponent_team_color=app_config.opponent_team_color,
        report=report,
    )
    radar_charts_data, radar_tables_data = prepare_radar_artifacts(
        doc_structure, home_name, away_name, radar_generator, exporter, scraper, app_config, report
    )
    shot_chart_data = generate_shot_chart_artifacts(scraper, session, exporter, app_config, report)

    doc_structure.extend(radar_tables_data)
    doc_title = build_doc_title(scraper.file_date)
    try:
        document_url = exporter.create_doc(
            doc_title,
            doc_structure,
            timeline_url=timeline_url,
            shot_chart_data=shot_chart_data,
            radar_charts_data=radar_charts_data,
        )
        report.document_url = document_url
        report.ok("Docs出力", doc_title)
    except Exception:
        logger.exception("Google Docs の生成に失敗しました")
        report.ng("Docs出力", doc_title)

    # 画像は Docs 挿入時点で Docs 側にコピーされるので、ここで公開を止める。
    if app_config.drive.revoke_public_links_after_export:
        revoked, failed = exporter.revoke_public_links()
        if revoked or failed:
            report.ok("公開リンク解除", f"解除 {revoked}件 / 失敗 {failed}件")
    else:
        report.skip("公開リンク解除", "設定で無効化されています（画像はURLを知れば誰でも閲覧可）")

    purged = exporter.purge_old_temp_files()
    if purged:
        report.ok("古い一時画像の整理", f"{purged}件をゴミ箱へ移動")

    report.emit()
    return report


if __name__ == "__main__":
    main()